# Model-Based / Model-Free Revaluation Task: V4 Analysis

V4 of the task (May 2026). Same MB/MF revaluation paradigm as V3 but with three task changes:

1. **Teleprompter / public-speaking threat induction.** A new phase added before structure learning that frames the upcoming task as having a public-speaking penalty contingent on shapes-game performance. Generates new fields: `Final_Earned_Speaking_Time_Sec`, `SpeakingTime_Log`, `Teleprompter_*`, `Threat_Induction_RT`.
2. **Query option grids restructured.** 4 options per state per query → **6 options per state per query** (12 trials per query). Values are MB/MF-discriminative for the assumed model. See log entry 2026-05-13.
3. **Revaluation briefings now show the decision tree.** Visual change only; doesn't affect downstream data structure.

## Task structure recap

1. **Structure Learning**: 32 forced trials; deterministic step 1, 75/25 common/rare at step 2.
2. **Value Learning (VL)**: 8 forced + 12 free-choice trials. MF "stage rewards" learned here: Feather=+5, Triangle=−5. All terminals=0 during VL.
3. **Revaluation 1** then **Query 1**, then **Revaluation 2** then **Query 2**. Each query = 12 trials (6 Feather offers + 6 Triangle offers).
4. **Ratings ×3** + **Surveys** (ERQ, Anxiety pre, Anxiety post).

## MB_EV table (assumed model — 75/25 step-2 stochasticity, optimal second-stage action, ±5 stage rewards)

| Query | Feather | Triangle |
|-------|--------:|---------:|
| 1     | −15     | +15      |
| 2     | −10     | +10      |

The only change from V3 is **(Q2, Feather) = 0 → −10**, because afterReval2 T1 went from −20 to −60.

## Pipeline (same as April-23 Modelling_AllChoices)

Descriptives → LMMs → RT analyses → Overthinking → CBM/HBI fitting → MF vs MB collapse → Mediation → Sensitivity → Recovery.

> ⚠ **N=20 in this export.** All inferential results are preliminary smoke-tests; modelling pipelines run but power is low. The April-23 logbook has the N=139 V3 numbers as a reference.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import linregress
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11

# ══════════════════════════════════════════════════════════════════════
# LOAD EXPORT  ── single source of truth for the input dataset.
# To switch datasets (e.g. the ~400-participant export), change
# DATA_FILE and DATASET_TAG below; nothing else needs editing.
# Older test-run exports live in prereg_3_testdrives/.
# ══════════════════════════════════════════════════════════════════════
DATA_FILE   = 'experiment_results_v4_1_export.csv'  # input CSV
DATASET_TAG = 'v4_1'                                # used in OUT_PREFIX (cell 2)

df = pd.read_csv(DATA_FILE)
df['subject_id'] = df['prolific_id'].fillna(df['firestore_uid'])
df = df[df['status'].str.upper() == 'COMPLETED'].reset_index(drop=True)
N_LOADED = len(df)
print(f'Loaded N = {N_LOADED} COMPLETED subjects from {DATA_FILE}  (DATASET_TAG={DATASET_TAG!r})')

# ══════════════════════════════════════════════════════════════════════
# SUBJECT FILTERING (folded in from V3 Generate_All_Files Steps 2-3)
# ══════════════════════════════════════════════════════════════════════
# Two filters, applied directly here so this notebook is the single
# source of truth for the filtering logic. After applying both filters,
# this cell ALSO writes filtered_data_<DATASET_TAG>.csv as a derived
# artifact for the Paul/RT_Analyses.ipynb companion notebook (see the
# to_csv call at the end of this cell). Paul never re-derives the
# exclusions; it just consumes what we write here.
#
# 1. Zig-zag inconsistency (P(chose fixed) curve should be monotonic
#    in offer value within each query x state cell). Sum the "drops"
#    across the 4 cells; exclude if total >= ZIGZAG_THRESHOLD.
# 2. Knowledge_Check_Accuracy > KNOWLEDGE_CHECK_MIN.
#
# NOT applying the V3 dropna() step: in V4 some columns are
# expected-NaN (e.g. Teleprompter_Silence_Triggered is NaN for
# participants who never triggered the silence warning) and
# total_time_minutes is unpopulated in the V4 export. A blanket
# dropna() would remove every subject. Per-column NaN handling
# happens downstream where it matters.
# ══════════════════════════════════════════════════════════════════════

ZIGZAG_THRESHOLD = 2.0
KNOWLEDGE_CHECK_MIN = 0.25

# Build long-format choice table for the zig-zag check
_long_rows = []
for _, row in df.iterrows():
    sid = row['subject_id']
    for q in [1, 2]:
        for t in range(1, 21):
            state = row.get(f'Query_{q}_T{t}_State')
            offer = row.get(f'Query_{q}_T{t}_FixedOffer')
            choice = row.get(f'Query_{q}_T{t}_Choice')
            if pd.isna(state) or pd.isna(offer) or pd.isna(choice):
                continue
            _long_rows.append({
                'subject_id': sid, 'Query': q, 'State': state,
                'FixedOffer': offer, 'ChoseFixed': 1 if choice == 0 else 0,
            })
_long = pd.DataFrame(_long_rows)

def _zigzag_score(sub_df):
    total = 0.0
    for _, g in sub_df.groupby(['Query', 'State']):
        curve = g.groupby('FixedOffer')['ChoseFixed'].mean().sort_index().values
        for i in range(len(curve) - 1):
            if curve[i] > curve[i+1]:
                total += curve[i] - curve[i+1]
    return total

zz = _long.groupby('subject_id', group_keys=False).apply(_zigzag_score)
zz_excluded = set(zz[zz >= ZIGZAG_THRESHOLD].index)

# Knowledge check filter
kc_excluded = set(df.loc[df['Knowledge_Check_Accuracy'] <= KNOWLEDGE_CHECK_MIN, 'subject_id'])

# Combine
all_excluded = zz_excluded | kc_excluded
filtering_log = pd.DataFrame({
    'subject_id': df['subject_id'],
    'zigzag_score': df['subject_id'].map(zz),
    'zigzag_excluded': df['subject_id'].isin(zz_excluded),
    'knowledge_check': df['Knowledge_Check_Accuracy'],
    'kc_excluded': df['subject_id'].isin(kc_excluded),
    'excluded': df['subject_id'].isin(all_excluded),
})

df = df[~df['subject_id'].isin(all_excluded)].reset_index(drop=True)
N_SUBJ = len(df)
print(f'Excluded zig-zag (>= {ZIGZAG_THRESHOLD}): {len(zz_excluded)}')
print(f'Excluded knowledge check (<= {KNOWLEDGE_CHECK_MIN}): {len(kc_excluded)}')
print(f'Final N after filtering: {N_SUBJ} (kept {N_SUBJ}/{N_LOADED})')

# ═════════════════════════════════════════════════════════════════════
# PREREG ALIGNMENT (2026-05-27): write the post-exclusion filtered df
# to CSV so the Paul/RT_Analyses.ipynb companion notebook can consume
# it without duplicating the exclusion logic. V4 remains the single
# source of truth for filtering; the CSV is a derived artifact.
# ═════════════════════════════════════════════════════════════════════
_filtered_csv = f'filtered_data_{DATASET_TAG}.csv'
df.to_csv(_filtered_csv, index=False)
print(f"Wrote {_filtered_csv} ({len(df)} rows) for RT_Analyses companion notebook.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIG (V4)
# ══════════════════════════════════════════════════════════════════════
#   FIT_ALL_TRIALS = True   → use every query trial
#   FIT_ALL_TRIALS = False  → use only anti-MF trials (the MB-vs-MF
#                             discriminative subset — for V4 this is
#                             roughly 18/24 trials by design)
#
# Output paths get a <DATASET_TAG>_<suffix> prefix (e.g. v4_1_all) so
# outputs from different exports don't collide. Old v4 test-run outputs
# are archived under prereg_3_testdrives/ and cbm_output_v4_all/.
# ══════════════════════════════════════════════════════════════════════
FIT_ALL_TRIALS = True

SEED = 42
np.random.seed(SEED)

FILTER_SUFFIX = "all" if FIT_ALL_TRIALS else "antimf"
FILTER_LABEL  = "ALL trials" if FIT_ALL_TRIALS else "ANTI-MF trials only"
OUT_PREFIX    = f"{DATASET_TAG}_{FILTER_SUFFIX}"   # e.g. v4_1_all
print(f"[CONFIG] FIT_ALL_TRIALS={FIT_ALL_TRIALS}  ({FILTER_LABEL})")
print(f"[CONFIG] SEED={SEED}")
print(f"[CONFIG] FILTER_SUFFIX='{FILTER_SUFFIX}'  OUT_PREFIX='{OUT_PREFIX}'")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# COMPUTE DERIVED VARIABLES & BUILD TRIAL-LEVEL DATA (V4)
# ══════════════════════════════════════════════════════════════════════
# Differences from V3 pipeline:
#   - 12 trials per query (not 8): range(1, 13)
#   - EV_MAP[(2,'Feather')] = -10 (was 0 in V3, because afterReval2 T1
#     changed from -20 to -60). All other EV_MAP entries unchanged.
#   - id_vars now carries V4 threat-induction columns so downstream
#     analyses can use Final_Earned_Speaking_Time_Sec etc.

# ── Survey subscales ──
erq_reappraisal_items = ['Survey_ERQ_Q1_Raw','Survey_ERQ_Q3_Raw','Survey_ERQ_Q5_Raw',
                         'Survey_ERQ_Q7_Raw','Survey_ERQ_Q8_Raw','Survey_ERQ_Q10_Raw']
erq_suppression_items = ['Survey_ERQ_Q2_Raw','Survey_ERQ_Q4_Raw',
                         'Survey_ERQ_Q6_Raw','Survey_ERQ_Q9_Raw']

df = df.copy()
df['ERQ_Reappraisal'] = df[erq_reappraisal_items].sum(axis=1)
df['ERQ_Suppression']  = df[erq_suppression_items].sum(axis=1)
df['somatic_anxiety']  = df['Survey_ANXIETY_2_Total_Score']
df['worry']            = df['Survey_ANXIETY_1_Total_Score']

# ── Ratings ──
for stim in ['Feather', 'Triangle']:
    for t in [1, 2, 3]:
        df[f'{stim}_R{t}'] = df[f'Rating_{t}_{stim}_Val']

# ── EV map (V4: only (2,Feather) differs from V3) ──
EV_MAP = {
    (1, 'Feather'): -15, (1, 'Triangle'): 15,
    (2, 'Feather'): -10, (2, 'Triangle'): 10,
}
MF_MAP = {'Feather': 5, 'Triangle': -5}

# ── Build trial-level dataframe ──
N_QUERY_TRIALS = 12   # V4: 6 Feather + 6 Triangle offers per query
rows = []
for _, subj in df.iterrows():
    sid = subj['subject_id']
    for q in [1, 2]:
        for t in range(1, N_QUERY_TRIALS + 1):
            pfx = f'Query_{q}_T{t}'
            state  = subj.get(f'{pfx}_State')
            choice = subj.get(f'{pfx}_Choice')
            offer  = subj.get(f'{pfx}_FixedOffer')
            rt     = subj.get(f'{pfx}_RT')
            if pd.isna(state) or pd.isna(choice) or pd.isna(offer):
                continue
            ev = EV_MAP.get((q, state), np.nan)
            mf = MF_MAP[state]
            ev_diff = ev - offer
            abs_ev_diff = np.abs(ev_diff)
            optimal = 1 if ev > offer else (0 if ev < offer else 1)
            correct = int(choice == optimal) if not np.isnan(optimal) else np.nan

            # PREREG: correct_canon scoring ∈ {0, 0.5, 1}
            #   1.0 if choice matches sign(EV − offer); 0.0 if wrong sign;
            #   0.5 if EV == offer (value-neutral, neither right nor wrong).
            if ev > offer:
                correct_canon = 1.0 if choice == 1 else 0.0
            elif ev < offer:
                correct_canon = 1.0 if choice == 0 else 0.0
            else:
                correct_canon = 0.5

            mf_says_gamble = 1 if mf > offer else (0 if mf < offer else np.nan)
            mb_says_gamble = 1 if ev > offer else (0 if ev < offer else np.nan)

            if pd.notna(mb_says_gamble):
                if pd.notna(mf_says_gamble) and mf_says_gamble != mb_says_gamble:
                    is_anti_mf = 1; chose_mb = int(choice == mb_says_gamble)
                elif pd.isna(mf_says_gamble):
                    is_anti_mf = 1; chose_mb = int(choice == mb_says_gamble)
                else:
                    is_anti_mf = 0; chose_mb = np.nan
            else:
                is_anti_mf = np.nan; chose_mb = np.nan

            log_rt = np.log(rt) if pd.notna(rt) and rt > 0 else np.nan

            rows.append({
                'subject_id': sid, 'query_num': q, 'trial': t,
                'trial_total': t + (N_QUERY_TRIALS if q == 2 else 0),
                'state': state, 'is_triangle': 1 if state == 'Triangle' else 0,
                'choice': int(choice), 'offer': offer, 'ev': ev, 'mf_value': mf,
                'ev_diff': ev_diff, 'abs_ev_diff': abs_ev_diff, 'correct': correct,
                'is_anti_mf': is_anti_mf, 'chose_mb': chose_mb,
                'correct_canon': correct_canon,
                'rt': rt, 'log_rt': log_rt,
                'somatic_anxiety': subj['somatic_anxiety'],
                'erq_reappraisal': subj['ERQ_Reappraisal'],
                'erq_suppression': subj['ERQ_Suppression'],
            })

trial_df = pd.DataFrame(rows)
trial_df = trial_df.dropna(subset=['correct'])

trial_df['within_query_ev_diff_z'] = trial_df.groupby(['query_num', 'state'])['ev_diff'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0)

anti_mf_df = trial_df[trial_df['is_anti_mf'] == 1].copy()

# ── Subject-level id_vars (now carries V4 threat-induction columns) ──
id_vars = df[['subject_id', 'somatic_anxiety', 'worry',
              'ERQ_Reappraisal', 'ERQ_Suppression',
              'Final_Earned_Speaking_Time_Sec', 'Teleprompter_Attempts',
              'Teleprompter_Silence_Triggered', 'Threat_Induction_RT']].copy()
id_vars = id_vars.rename(columns={'Final_Earned_Speaking_Time_Sec': 'earned_time'})

print(f'Trial-level data: {len(trial_df)} obs from {trial_df.subject_id.nunique()} subjects')
print(f'Anti-MF trials:   {trial_df.is_anti_mf.sum():.0f} / {trial_df.is_anti_mf.notna().sum()}')
print(f'RT available:     {trial_df.log_rt.notna().sum()} trials with valid log(RT)')
print(f'N subjects:       {trial_df.subject_id.nunique()}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MODEL-AGNOSTIC mb_score CLASSIFIER  (preregistered primary partition)
# ══════════════════════════════════════════════════════════════════════
# Per the prereg (Variables / Model-agnostic model-based classification):
#
#   mb_score_s  =  P(chose MB | anti-MF query trial in stim s)
#
# Subject is MB-classified if  mb_score_F > 0.5  OR  mb_score_T > 0.5
# (lenient, either-stim, above-chance rule).
#
# This cell builds `mb_score_df`, a per-subject DataFrame with columns
# {mb_score_F, mb_score_T, n_anti_F, n_anti_T, mb_classif_agnostic}.
# H1, H2, H3 confirmatory tests partition on `mb_classif_agnostic`.
# The CBM/HBI classification (built later in section 8) is used as the
# convergent-validity check, not the primary partition.
# ══════════════════════════════════════════════════════════════════════

# Restrict to anti-MF trials with a valid chose_mb
_anti = trial_df.dropna(subset=['chose_mb'])
_anti = _anti[_anti['is_anti_mf'] == 1].copy()

# Per-subject × per-stim mean of chose_mb
_mb = (_anti
       .groupby(['subject_id', 'state'])['chose_mb']
       .agg(['mean', 'count'])
       .rename(columns={'mean': 'mb_score', 'count': 'n_anti'})
       .reset_index())

# Pivot to one row per subject with F and T columns
_pivot = _mb.pivot(index='subject_id', columns='state',
                   values=['mb_score', 'n_anti'])
_pivot.columns = [f'{a}_{b[0]}' for a, b in _pivot.columns]   # e.g. mb_score_F
_pivot = _pivot.reset_index()

# Make sure both columns exist even when a stim has no anti-MF data
for col in ['mb_score_F', 'mb_score_T', 'n_anti_F', 'n_anti_T']:
    if col not in _pivot.columns:
        _pivot[col] = np.nan

# Apply the lenient OR-rule: MB if either stim above chance
_pivot['mb_classif_agnostic'] = (
    (_pivot['mb_score_F'] > 0.5) | (_pivot['mb_score_T'] > 0.5)
)

mb_score_df = _pivot[['subject_id',
                      'mb_score_F', 'mb_score_T',
                      'n_anti_F', 'n_anti_T',
                      'mb_classif_agnostic']].copy()

# Summary print
_n_mb  = int(mb_score_df['mb_classif_agnostic'].sum())
_n_tot = len(mb_score_df)
_pct   = 100 * _n_mb / _n_tot if _n_tot else float('nan')
print(f'Model-agnostic classifier: {_n_mb}/{_n_tot} subjects MB ({_pct:.1f}%)')
print(f'  mb_score_F: mean={mb_score_df["mb_score_F"].mean():.3f}, '
      f'median={mb_score_df["mb_score_F"].median():.3f}, '
      f'n with data = {mb_score_df["mb_score_F"].notna().sum()}')
print(f'  mb_score_T: mean={mb_score_df["mb_score_T"].mean():.3f}, '
      f'median={mb_score_df["mb_score_T"].median():.3f}, '
      f'n with data = {mb_score_df["mb_score_T"].notna().sum()}')
print(f'  Subjects classified MB on Feather only:  '
      f'{int(((mb_score_df["mb_score_F"]>0.5) & ~(mb_score_df["mb_score_T"]>0.5)).sum())}')
print(f'  Subjects classified MB on Triangle only: '
      f'{int((~(mb_score_df["mb_score_F"]>0.5) & (mb_score_df["mb_score_T"]>0.5)).sum())}')
print(f'  Subjects classified MB on both stims:    '
      f'{int(((mb_score_df["mb_score_F"]>0.5) & (mb_score_df["mb_score_T"]>0.5)).sum())}')


---
## 1.5 · V4-specific descriptives (threat induction)

These columns don't exist in V3 data. They capture the silent earned-speaking-time accumulation driven by sub-optimal MB choices.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# V4: earned-speaking-time + threat-induction descriptives
# ══════════════════════════════════════════════════════════════════════
import ast

# ── 1) Final earned speaking time distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(id_vars['earned_time'].dropna(), bins=12, color='#dc2626', edgecolor='white')
ax.axvline(30,  ls='--', color='gray', label='base (30 s)')
ax.axvline(300, ls='--', color='red',  label='ceiling (300 s)')
ax.set_xlabel('Final earned speaking time (s)')
ax.set_ylabel('Subjects')
ax.set_title(f'Distribution of earned time (N={id_vars.earned_time.notna().sum()})')
ax.legend()

# ── 2) earned_time vs anxiety ──
ax = axes[1]
plot_df = id_vars.dropna(subset=['earned_time','somatic_anxiety'])
ax.scatter(plot_df['somatic_anxiety'], plot_df['earned_time'], alpha=0.7, color='#1e3c72')
if len(plot_df) > 2:
    r, p = stats.spearmanr(plot_df['somatic_anxiety'], plot_df['earned_time'])
    ax.set_title(f'Earned time × Somatic Anxiety   ρ={r:.2f}  p={p:.3f}')
ax.set_xlabel('STAI-S (post)')
ax.set_ylabel('Final earned speaking time (s)')

plt.tight_layout()
plt.savefig(f'fig_{OUT_PREFIX}_earned_time.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 3) Parse SpeakingTime_Log → penalty breakdown by phase ──
def parse_log(s):
    if pd.isna(s) or not isinstance(s, str): return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

log_records = []
for _, r in df.iterrows():
    sid = r['subject_id']
    for entry in parse_log(r.get('SpeakingTime_Log', '[]')):
        log_records.append({'subject_id': sid, **entry})
log_df = pd.DataFrame(log_records)

if len(log_df):
    print('Penalty events breakdown:')
    print(log_df.groupby('phase')['added'].agg(['count','sum']).to_string())
    print()
    print('Per-subject penalty events (mean ± sd):')
    per_subj = log_df.groupby('subject_id')['added'].agg(['count','sum'])
    print(per_subj.describe().to_string())
else:
    print('No penalty events in SpeakingTime_Log.')

# ── 4) Suboptimal flags from the per-trial Query_*_Suboptimal columns ──
subopt_cols = [c for c in df.columns if 'Suboptimal' in c]
df['n_suboptimal'] = df[subopt_cols].sum(axis=1)
df['n_suboptimal_q1'] = df[[c for c in subopt_cols if c.startswith('Query_1')]].sum(axis=1)
df['n_suboptimal_q2'] = df[[c for c in subopt_cols if c.startswith('Query_2')]].sum(axis=1)
print()
print(f"Suboptimal choices per subject (out of {N_QUERY_TRIALS*2}): "
      f"mean={df.n_suboptimal.mean():.2f}  median={df.n_suboptimal.median()}  range={df.n_suboptimal.min()}-{df.n_suboptimal.max()}")

# Pass the per-subject counts back into id_vars
id_vars = id_vars.merge(df[['subject_id','n_suboptimal','n_suboptimal_q1','n_suboptimal_q2']],
                        on='subject_id', how='left')


---
## 1 · Descriptives

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1A  RATINGS ACROSS TIME
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

for i, stim in enumerate(['Feather', 'Triangle']):
    ax = axes[i]
    vals = df[[f'{stim}_R1', f'{stim}_R2', f'{stim}_R3']].values
    means = vals.mean(axis=0)
    sems  = vals.std(axis=0) / np.sqrt(len(df))
    timepoints = ['Post-MF\n(before reval)', 'Post-Reval 1', 'Post-Reval 2']
    
    # Spaghetti plot (faint individual lines)
    for row in vals:
        ax.plot([0,1,2], row, color='grey', alpha=0.08, lw=0.5)
    
    ax.errorbar([0,1,2], means, yerr=1.96*sems, marker='o', color='C0' if stim=='Feather' else 'C3',
                lw=2.5, capsize=5, markersize=8, zorder=5)
    ax.axhline(MF_MAP[stim], color='grey', ls='--', alpha=0.5, label=f'MF value ({MF_MAP[stim]})')
    ax.axhline(0, color='black', ls='-', alpha=0.2, lw=0.5)
    ax.set_xticks([0,1,2])
    ax.set_xticklabels(timepoints)
    ax.set_title(f'{stim} Ratings', fontsize=13, fontweight='bold')
    ax.set_ylabel('Rating') if i==0 else None
    ax.legend(fontsize=9)

fig.suptitle('Rating Trajectories Over Time', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print summary
for stim in ['Feather', 'Triangle']:
    print(f'\n{stim}:')
    for t, label in zip([1,2,3], ['Post-MF','Post-Reval1','Post-Reval2']):
        col = f'{stim}_R{t}'
        print(f'  {label}: M = {df[col].mean():.2f}, SD = {df[col].std():.2f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1B  QUERY PERFORMANCE: P(GAMBLE) BY EV DIFFERENCE
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for qi, q in enumerate([1, 2]):
    ax = axes[qi]
    sub = trial_df[trial_df['query_num'] == q]
    
    for stim, color in [('Feather','C0'), ('Triangle','C3')]:
        s = sub[sub.state == stim]
        grouped = s.groupby('ev_diff')['choice'].agg(['mean','sem','count']).reset_index()
        ax.errorbar(grouped['ev_diff'], grouped['mean'], yerr=1.96*grouped['sem'],
                    marker='o', label=stim, color=color, capsize=3)
    
    ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
    ax.axvline(0, color='grey', ls='--', alpha=0.4)
    ax.set_xlabel('EV difference (gamble EV − fixed offer)')
    ax.set_ylabel('P(choose gamble)') if qi==0 else None
    ax.set_title(f'Query {q} (Revaluation {q})', fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_ylim(-0.05, 1.05)

fig.suptitle('Choice Curves by EV Difference', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Overall accuracy
print('\nOverall accuracy (chose optimally given MB EV):')
for q in [1, 2]:
    sub = trial_df[trial_df['query_num'] == q]
    for stim in ['Feather', 'Triangle']:
        s = sub[sub.state == stim]
        print(f'  Query {q} / {stim}: {s.correct.mean():.3f} (n_trials={len(s)})')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 1C  ACCURACY BY QUERY × STATE (BAR PLOT)
# ══════════════════════════════════════════════════════════════════════

acc_summary = trial_df.groupby(['query_num','state'])['correct'].agg(['mean','sem']).reset_index()

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(2)
w = 0.35
for i, stim in enumerate(['Feather','Triangle']):
    sub = acc_summary[acc_summary.state == stim]
    ax.bar(x + i*w, sub['mean'], w, yerr=1.96*sub['sem'],
           label=stim, color=['C0','C3'][i], capsize=5, alpha=0.85)

ax.set_xticks(x + w/2)
ax.set_xticklabels(['Query 1\n(Reval 1)', 'Query 2\n(Reval 2)'])
ax.set_ylabel('Accuracy (MB-optimal choice)')
ax.set_title('MB-Optimal Accuracy by Query × Stimulus', fontweight='bold')
ax.legend()
ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 2 · Linear Mixed Models

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# LMM UTILITIES (statsmodels implementation)
# ══════════════════════════════════════════════════════════════════════
import re
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM


def _parse_lme4_formula(formula_str):
    """
    Parse an lme4-style formula string into fixed and random parts.

    Supports syntax like:
        y ~ x1 + x2 + (1 + x1 | group)
        y ~ 1 + x1 + (x1 | group)
        y ~ (1 + x1 + x2 | group)
        y ~ x1 + (0 + x1 | group)

    Rules
    -----
    - Terms inside (...|group) define RANDOM effects for that group.
      '1' inside means random intercept; '0' means suppress random intercept.
    - Terms OUTSIDE (...|...) define FIXED effects.
      '1' outside means fixed intercept (default); '0' means suppress it.
    - A term present ONLY inside the random part gets no fixed-effect column.
    - A term present in BOTH gets both a fixed slope and a random slope.

    Returns
    -------
    outcome        : str   — left-hand side variable name
    fixed_intercept: bool  — whether to include a fixed intercept
    fixed_terms    : list  — non-intercept fixed-effect term names
    random_spec    : dict  — {group_col: {'intercept': bool, 'slopes': [str]}}
    """
    formula_str = formula_str.strip()

    # ── Split LHS and RHS ─────────────────────────────────────────────
    lhs, rhs = formula_str.split('~', 1)
    outcome = lhs.strip()
    rhs = rhs.strip()

    # ── Extract all (terms | group) blocks ────────────────────────────
    random_blocks = re.findall(r'\(([^)]+)\)', rhs)

    # Remove random blocks from RHS to get fixed-only part
    fixed_rhs = re.sub(r'\([^)]+\)', '', rhs)
    # Clean up leftover operators
    fixed_rhs = re.sub(r'\+\s*\+', '+', fixed_rhs)
    fixed_rhs = fixed_rhs.strip().strip('+').strip()

    # ── Parse fixed terms ─────────────────────────────────────────────
    fixed_intercept = True   # default: include intercept
    fixed_terms = []

    if fixed_rhs:
        for tok in re.split(r'\+', fixed_rhs):
            tok = tok.strip()
            if tok in ('', '1'):
                fixed_intercept = True
            elif tok == '0':
                fixed_intercept = False
            else:
                fixed_terms.append(tok)

    # ── Parse random blocks ────────────────────────────────────────────
    # Each block: "1 + x1 + x2 | group"  or  "0 + x1 | group"
    random_spec = {}

    for block in random_blocks:
        if '|' not in block:
            raise ValueError(f"Random block '{block}' missing '|' separator.")
        re_terms_str, group = block.rsplit('|', 1)
        group = group.strip()

        re_intercept = True
        re_slopes = []

        for tok in re.split(r'\+', re_terms_str):
            tok = tok.strip()
            if tok in ('', '1'):
                re_intercept = True
            elif tok == '0':
                re_intercept = False
            else:
                re_slopes.append(tok)

        random_spec[group] = {
            'intercept': re_intercept,
            'slopes':    re_slopes,
        }

    return outcome, fixed_intercept, fixed_terms, random_spec


def _build_statsmodels_inputs(outcome, fixed_intercept, fixed_terms, random_spec):
    """
    Translate parsed formula components into the strings/dicts that
    BinomialBayesMixedGLM.from_formula() expects.

    BinomialBayesMixedGLM takes:
      - fixed_formula  : standard patsy formula for fixed effects
      - random_formula : dict {label: patsy_formula_string}
                         Each entry is one variance component (its own SD).
                         Intercept and each slope get separate entries.

    Returns
    -------
    fixed_formula  : str
    random_formula : dict
    """
    # Fixed formula
    fixed_rhs_parts = []
    if not fixed_intercept:
        fixed_rhs_parts.append('0')
    fixed_rhs_parts += fixed_terms
    fixed_formula = f"{outcome} ~ {' + '.join(fixed_rhs_parts) or '1'}"

    # Random formula — one entry per variance component
    # Intercept and each slope get their own entry (independent SDs)
    random_formula = {}
    for group, spec in random_spec.items():
        if spec['intercept']:
            random_formula[f'{group}_intercept'] = f'0 + C({group})'
        for slope in spec['slopes']:
            random_formula[f'{group}_{slope}'] = f'0 + C({group}):{slope}'

    return fixed_formula, random_formula


def logistic_lmm(df_long, formula, group_col=None, verbose=True):
    """
    Fit a logistic mixed-effects model using an lme4-style formula.

    The formula fully specifies what is fixed and what is random.
    No separate fixed_cols / group_col arguments needed.

    Supported syntax examples
    -------------------------
    'y ~ x1 + x2 + (1 + x1 | subject)'
        fixed intercept, fixed x1+x2, random intercept + random slope x1

    'y ~ x1 + (x2 | subject)'
        fixed intercept (default), fixed x1, random slope x2 only

    'y ~ 0 + x1 + (1 | subject)'
        NO fixed intercept, fixed x1, random intercept only

    'y ~ (1 + x1 + x2 | subject)'
        fixed intercept (default from patsy), random intercept + slopes x1 x2
        no extra fixed slopes (x1, x2 only modelled as random)

    Parameters
    ----------
    df_long   : pd.DataFrame — trial-level long-format data
    formula   : str          — lme4-style formula (see above)
    group_col : str or None  — ignored; kept for backwards compatibility
    verbose   : bool         — print results table

    Returns
    -------
    dict with keys:
        'results'        : pd.DataFrame — Parameter, Estimate, SE, z, p
        'beta'           : np.ndarray   — posterior means (fixed effects)
        'se'             : np.ndarray   — posterior SDs  (fixed effects)
        'sigma_u'        : dict         — {component_label: SD} per random term
        'random_effects' : np.ndarray   — all random coefficient posteriors
        'converged'      : bool
        'formula'        : str
        'model'          : fitted BayesMixedGLMResults object
    """
    if verbose:
        print(f'\n{"="*65}')
        print(f'Logistic GLMM: {formula}')
        print(f'{"="*65}')

    # ── Parse formula ──────────────────────────────────────────────────
    outcome, fixed_intercept, fixed_terms, random_spec = _parse_lme4_formula(formula)

    # ── Build statsmodels inputs ───────────────────────────────────────
    fixed_formula, random_formula = _build_statsmodels_inputs(
        outcome, fixed_intercept, fixed_terms, random_spec
    )

    if verbose:
        print(f'  Fixed formula : {fixed_formula}')
        print(f'  Random terms  : {list(random_formula.keys())}')

    n_obs  = len(df_long)
    n_subj = {g: df_long[g].nunique() for g in random_spec}

    # ── Fit ────────────────────────────────────────────────────────────
    model  = BinomialBayesMixedGLM.from_formula(
        fixed_formula,
        random_formula,
        data=df_long
    )
    result = model.fit_vb()

    # ── Fixed effects ──────────────────────────────────────────────────
    beta  = np.asarray(result.fe_mean)
    se    = np.asarray(result.fe_sd)
    names = list(model.exog_names)

    z      = beta / se
    p_vals = 2 * (1 - stats.norm.cdf(np.abs(z)))

    results_df = pd.DataFrame({
        'Parameter': names,
        'Estimate':  beta,
        'SE':        se,
        'z':         z,
        'p':         p_vals
    })

    # ── Variance components ────────────────────────────────────────────
    # vcp_mean is a vector of log(SD), one per entry in random_formula,
    # in the same order as random_formula keys
    vc_labels = list(random_formula.keys())
    sigma_u = {
        label: float(np.exp(vcp))
        for label, vcp in zip(vc_labels, result.vcp_mean)
    }

    re_values = np.asarray(result.vc_mean)

    # ── Print ──────────────────────────────────────────────────────────
    if verbose:
        print(f'\nN obs = {n_obs}')
        for g, n in n_subj.items():
            print(f'N {g} = {n}')
        print(f'\nRandom effect SDs:')
        for label, sd in sigma_u.items():
            print(f'  {label:35s}  σ={sd:.4f}')
        print(f'\nFixed Effects:')
        for _, row in results_df.iterrows():
            sig = ('***' if row['p'] < .001 else
                   '**'  if row['p'] < .01  else
                   '*'   if row['p'] < .05  else '')
            print(f"  {row['Parameter']:25s}  β={row['Estimate']:+.4f}"
                  f"  SE={row['SE']:.4f}  z={row['z']:+.3f}"
                  f"  p={row['p']:.4f} {sig}")
        print()
        print(result.summary())

    return {
        'results':        results_df,
        'beta':           beta,
        'se':             se,
        'sigma_u':        sigma_u,
        'random_effects': re_values,
        'converged':      True,
        'formula':        formula,
        'model':          result
    }

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2B  LMM: Query 2 only — Accuracy difference Feather vs Triangle
# ══════════════════════════════════════════════════════════════════════

q2_df = trial_df[trial_df['query_num'] == 2].copy()
q2_df['is_triangle'] = (q2_df['state'] == 'Triangle').astype(float)
q2_df['trial_total'] = q2_df['trial_total'] - 1
lmm_q2 = logistic_lmm(
    q2_df,
    'correct ~ 1+abs_ev_diff+is_triangle+trial_total+(1+abs_ev_diff+is_triangle+trial_total|subject_id)',
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2C  LMM: correct ~ ev_diff + query + (1|subject)
# ══════════════════════════════════════════════════════════════════════

lmm_ev_df = trial_df.copy()
lmm_ev_df['query_c'] = lmm_ev_df['query_num'] - 1.5

lmm_ev = logistic_lmm(
    lmm_ev_df,
    'correct ~ 1+abs_ev_diff + query_c + (1+abs_ev_diff + query_c|subject_id)',
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2D  LMM: Anti-MF trials — both queries
#     chose_mb ~ query + abs_ev_diff + is_triangle + (1|subject)
# ══════════════════════════════════════════════════════════════════════

anti_mf_lmm = trial_df[trial_df['is_anti_mf'].fillna(0) == 1].copy()
anti_mf_lmm = anti_mf_lmm.dropna(subset=['chose_mb'])
anti_mf_lmm['query_c'] = anti_mf_lmm['query_num'] - 1.5
anti_mf_lmm['is_triangle'] = (anti_mf_lmm['state'] == 'Triangle').astype(float)

print(f'Anti-MF trials: {len(anti_mf_lmm)} from {anti_mf_lmm.subject_id.nunique()} subjects')
print(f'  Query 1: {len(anti_mf_lmm[anti_mf_lmm.query_num==1])}')
print(f'  Query 2: {len(anti_mf_lmm[anti_mf_lmm.query_num==2])}')
print(f'  Mean chose_mb Q1: {anti_mf_lmm[anti_mf_lmm.query_num==1].chose_mb.mean():.3f}')
print(f'  Mean chose_mb Q2: {anti_mf_lmm[anti_mf_lmm.query_num==2].chose_mb.mean():.3f}')

lmm_anti = logistic_lmm(
    anti_mf_lmm,
    'chose_mb ~ 1+query_c+abs_ev_diff+is_triangle+(1+query_c+abs_ev_diff+is_triangle|subject_id)',
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 2E  LMM: Anti-MF trials — Query 1 only
# ══════════════════════════════════════════════════════════════════════

anti_mf_q1 = trial_df[trial_df['is_anti_mf'].fillna(0) == 1].copy()
anti_mf_q1 = anti_mf_q1.dropna(subset=['chose_mb'])
anti_mf_q1 = anti_mf_q1[anti_mf_q1['query_num'] == 1]
anti_mf_q1['ev_diff_z'] = (anti_mf_q1['ev_diff'] - anti_mf_q1['ev_diff'].mean()) / anti_mf_q1['ev_diff'].std()
anti_mf_q1['is_triangle'] = (anti_mf_q1['state'] == 'Triangle').astype(float)

print(f'Anti-MF Q1 trials: {len(anti_mf_q1)} from {anti_mf_q1.subject_id.nunique()} subjects')
print(f'  Mean chose_mb: {anti_mf_q1.chose_mb.mean():.3f}')

lmm_anti_q1 = logistic_lmm(
    anti_mf_q1,
    'chose_mb ~ 1+abs_ev_diff+is_triangle+(1+abs_ev_diff+is_triangle|subject_id)',
)

#  ANTI-MF PERFORMANCE ACROSS QUERIES — DESCRIPTIVE

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Overall anti-MF performance by query
ax = axes[0]
anti_summary = anti_mf_df.groupby(['query_num','state']).agg(
    mean_mb=('chose_mb','mean'),
    sem_mb=('chose_mb','sem')
).reset_index()

x = np.arange(2)
w = 0.35
for i, stim in enumerate(['Feather','Triangle']):
    s = anti_summary[anti_summary.state == stim]
    ax.bar(x + i*w, s['mean_mb'], w, yerr=1.96*s['sem_mb'],
           label=stim, color=['C0','C3'][i], capsize=5, alpha=0.85)

ax.set_xticks(x + w/2)
ax.set_xticklabels(['Query 1 (Reval 1)', 'Query 2 (Reval 2)'])
ax.set_ylabel('P(chose MB over MF)')
ax.set_title('Anti-MF Trial Performance', fontweight='bold')
ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
ax.legend()
ax.set_ylim(0, 1)

# Panel 2: By anxiety median split
ax = axes[1]
med_anx = trial_df['somatic_anxiety'].median()
anti_mf_df['anx_group'] = np.where(anti_mf_df['somatic_anxiety'] >= med_anx, 'High Somatic Anxiety', 'Low Somatic Anxiety')

anx_summary = anti_mf_df.groupby(['query_num','anx_group']).agg(
    mean_mb=('chose_mb','mean'),
    sem_mb=('chose_mb','sem')
).reset_index()

for i, ag in enumerate(['Low Somatic Anxiety', 'High Somatic Anxiety']):
    s = anx_summary[anx_summary.anx_group == ag]
    ax.bar(x + i*w, s['mean_mb'], w, yerr=1.96*s['sem_mb'],
           label=ag, color=['C4','C1'][i], capsize=5, alpha=0.85)

ax.set_xticks(x + w/2)
ax.set_xticklabels(['Query 1 (Reval 1)', 'Query 2 (Reval 2)'])
ax.set_ylabel('P(chose MB over MF)')
ax.set_title('Anti-MF by Somatic Anxiety (Median Split)', fontweight='bold')
ax.axhline(0.5, color='grey', ls='--', alpha=0.4)
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## 5 · Reaction Time Analyses

All RT analyses use **log(RT)**. N = 139 (no exclusions).

### Theoretical motivation
Larger |EV difference| between the gamble EV and the fixed offer means the correct choice is more obvious — there is less to gain from deliberation. Therefore:
1. **RT should be *faster* when |EV diff| is large** — obvious choices need less thought
2. **RT should be *slower* when |EV diff| is small** — close calls demand more deliberation
3. **RT should improve accuracy, but primarily when |EV diff| is large** — deliberation is productive when stakes are high and there's a discernible correct answer

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5A  |EV diff| → log(RT) and RT × |EV diff| → accuracy
# ══════════════════════════════════════════════════════════════════════

# Work with anti-MF trials that have valid RT
rt_df = anti_mf_df[anti_mf_df['log_rt'].notna()].copy()
print(f'Anti-MF trials with valid log(RT): {len(rt_df)} from {rt_df.subject_id.nunique()} subjects')

# ── Standardize predictors ──
rt_df['abs_ev_diff_z'] = (rt_df['abs_ev_diff'] - rt_df['abs_ev_diff'].mean()) / rt_df['abs_ev_diff'].std()
rt_df['log_rt_z']      = (rt_df['log_rt'] - rt_df['log_rt'].mean()) / rt_df['log_rt'].std()
rt_df['anxiety_z']     = (rt_df['somatic_anxiety'] - rt_df['somatic_anxiety'].mean()) / rt_df['somatic_anxiety'].std()
rt_df['erq_r_z']       = (rt_df['erq_reappraisal'] - rt_df['erq_reappraisal'].mean()) / rt_df['erq_reappraisal'].std()
rt_df['query_c']       = rt_df['query_num'] - 1.5

# ══════════════════════════════════════════════════════════════════════
# MODEL 1: log(RT) ~ |EV diff|
# ══════════════════════════════════════════════════════════════════════
print('\n' + '═' * 70)
print('MODEL 1: |EV diff| → log(RT)')
print('═' * 70)

# Hard vs Easy median split
median_abs = rt_df['abs_ev_diff'].median()
rt_df['difficulty'] = np.where(rt_df['abs_ev_diff'] <= median_abs, 'Hard', 'Easy')

cell_rt = rt_df.groupby('difficulty').agg(
    log_rt_mean=('log_rt', 'mean'), log_rt_sem=('log_rt', 'sem'),
    n=('log_rt', 'count')
).reset_index()
for _, row in cell_rt.iterrows():
    print(f'  {row["difficulty"]:>6s}: log_rt = {row["log_rt_mean"]:.3f} ± {row["log_rt_sem"]:.3f} (n={int(row["n"])})')

hard_rt = rt_df[rt_df['difficulty'] == 'Hard']['log_rt']
easy_rt = rt_df[rt_df['difficulty'] == 'Easy']['log_rt']
t_rt, p_rt = stats.ttest_ind(hard_rt, easy_rt, equal_var=False)
d_rt = (hard_rt.mean() - easy_rt.mean()) / np.sqrt((hard_rt.var() + easy_rt.var()) / 2)
print(f'\n  Hard vs Easy: t={t_rt:.3f}, p={p_rt:.4f}, d={d_rt:.3f}')

# Trial-level Spearman
r_ev_rt, p_ev_rt = stats.spearmanr(rt_df['abs_ev_diff'], rt_df['log_rt'])
print(f'  |EV diff| × log_rt (Spearman): r={r_ev_rt:.3f}, p={p_ev_rt:.4f}')

# OLS with clustered SEs
print('\n--- OLS: log_rt ~ abs_ev_diff_z + is_triangle + query_c (clustered SEs) ---')
ols_data = rt_df[['log_rt', 'abs_ev_diff_z', 'is_triangle', 'query_c', 'subject_id']].dropna()
X_ols = sm.add_constant(ols_data[['abs_ev_diff_z', 'is_triangle', 'query_c']])
ols_model = sm.OLS(ols_data['log_rt'], X_ols).fit(cov_type='cluster',
                                                      cov_kwds={'groups': ols_data['subject_id']})
print(ols_model.summary2().tables[1])

# ══════════════════════════════════════════════════════════════════════
# MODEL 2: accuracy ~ log_rt × |EV diff|
# RT helps decisions BUT only when |EV diff| is high
# ══════════════════════════════════════════════════════════════════════
print('\n' + '═' * 70)
print('MODEL 2: accuracy ~ log_rt × |EV diff|')
print('═' * 70)

rt_df['log_rt_x_abs_ev'] = rt_df['log_rt_z'] * rt_df['abs_ev_diff_z']

lmm_rt_choice = logistic_lmm(
    rt_df,
    'chose_mb ~ 1 + log_rt_z + abs_ev_diff_z + log_rt_x_abs_ev '
    '+ is_triangle + query_c + (1 | subject_id)'
)

---
## 6 · Overthinking Analysis (Condensed)

**Model-agnostic approach**: Split trials by |EV diff| (median → Hard/Easy) and subjects by anxiety (median → High/Low).

1. **Log RT** by difficulty × anxiety
2. **P(Chose MB)** by difficulty × anxiety
3. **Overthinking metrics**: original index + alternate definition

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 6A  MODEL-AGNOSTIC 2×2 PLOTS: Difficulty × Somatic Anxiety
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = '#FAFBFC'

# ── Setup: difficulty split & anxiety split on anti-MF trials ──
anti_ot = anti_mf_df[anti_mf_df['log_rt'].notna()].copy()
median_diff = anti_ot['abs_ev_diff'].median()
anti_ot['difficulty'] = np.where(anti_ot['abs_ev_diff'] <= median_diff, 'Hard', 'Easy')

median_anx = anti_ot.groupby('subject_id')['somatic_anxiety'].first().median()
anti_ot['anx_group'] = anti_ot['subject_id'].map(
    anti_ot.groupby('subject_id')['somatic_anxiety'].first().apply(
        lambda a: 'High' if a >= median_anx else 'Low'))

n_subj = anti_ot['subject_id'].nunique()
print(f'Overthinking analysis: N = {n_subj}, {len(anti_ot)} anti-MF trials with RT')

# ── Cell means ──
cell_means = anti_ot.groupby(['anx_group', 'difficulty']).agg(
    log_rt_mean=('log_rt', 'mean'), log_rt_sem=('log_rt', 'sem'),
    mb_mean=('chose_mb', 'mean'), mb_sem=('chose_mb', 'sem'),
    n=('chose_mb', 'count'),
).reset_index()

print('\nCell means (Somatic Anxiety × Difficulty):')
for _, row in cell_means.iterrows():
    print(f'  {row["anx_group"]:>4s} ANX, {row["difficulty"]:>4s}: '
          f'logRT={row["log_rt_mean"]:.3f}±{row["log_rt_sem"]:.3f}  '
          f'P(MB)={row["mb_mean"]:.3f}±{row["mb_sem"]:.3f}  n={int(row["n"])}')

# ══════════════════════════════════════════════════════════════════════
# PLOT 1: Log RT by Difficulty × Somatic Anxiety (model-agnostic)
# ══════════════════════════════════════════════════════════════════════
ANX_COLORS = {'Low': '#3498DB', 'High': '#E74C3C'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5), facecolor=BG_COLOR)

for ax, metric, sem_col, ylabel, title in [
    (ax1, 'log_rt_mean', 'log_rt_sem', 'Log RT', 'Deliberation Time (Log RT)'),
    (ax2, 'mb_mean',     'mb_sem',     'P(Chose MB)', 'MB Choice Rate')]:
    ax.set_facecolor(BG_COLOR)
    x_order = {'Hard': 0, 'Easy': 1}
    for anx in ['Low', 'High']:
        sub = cell_means[cell_means['anx_group'] == anx].copy()
        sub['x'] = sub['difficulty'].map(x_order)
        sub = sub.sort_values('x')
        offset = -0.05 if anx == 'Low' else 0.05
        ax.errorbar(sub['x'] + offset, sub[metric], yerr=1.96 * sub[sem_col],
                    color=ANX_COLORS[anx], marker='o', lw=2.5, capsize=5,
                    ms=9, label=f'{anx} Somatic Anxiety', markeredgecolor='white', markeredgewidth=1)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Hard\n(low |ΔEV|)', 'Easy\n(high |ΔEV|)'])
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=10)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    if metric == 'mb_mean': ax.axhline(0.5, color='gray', ls=':', lw=0.8)

fig.suptitle('Model-Agnostic: Difficulty × Somatic Anxiety on Anti-MF Trials', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"fig_overthinking_condensed_{OUT_PREFIX}.png", dpi=200, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()

# ── Interaction tests ──
from scipy.stats import mannwhitneyu
for metric_name, metric_col in [('Log RT', 'log_rt'), ('P(MB)', 'chose_mb')]:
    hard_high = anti_ot[(anti_ot['difficulty']=='Hard') & (anti_ot['anx_group']=='High')][metric_col]
    hard_low  = anti_ot[(anti_ot['difficulty']=='Hard') & (anti_ot['anx_group']=='Low')][metric_col]
    easy_high = anti_ot[(anti_ot['difficulty']=='Easy') & (anti_ot['anx_group']=='High')][metric_col]
    easy_low  = anti_ot[(anti_ot['difficulty']=='Easy') & (anti_ot['anx_group']=='Low')][metric_col]
    # Difficulty effect by anxiety group
    t_h, p_h = stats.ttest_ind(hard_high, easy_high, equal_var=False)
    t_l, p_l = stats.ttest_ind(hard_low, easy_low, equal_var=False)
    print(f'\n{metric_name} — Difficulty effect:')
    print(f'  High ANX: Hard vs Easy t={t_h:.2f}, p={p_h:.4f}')
    print(f'  Low  ANX: Hard vs Easy t={t_l:.2f}, p={p_l:.4f}')
    # Anxiety effect on hard trials
    t_a, p_a = stats.ttest_ind(hard_high, hard_low, equal_var=False)
    print(f'  Hard trials: High vs Low ANX t={t_a:.2f}, p={p_a:.4f}')

---
## 8 · RL Model Fitting (CBM: Hierarchical Bayesian)

Using [Piray et al. (2019)](https://github.com/payampiray/cbm_python) — hierarchical Bayesian inference with Laplace approximation for concurrent model fitting and comparison.

### Choice-only models (anti-MF trials, predict chose_mb)

| # | Model | Parameters |
|---|-------|-----------|
| 1 | MF-only (fixed MF values) | beta |
| 2 | MF + learning rate | beta, alpha |
| 3 | MB-only | beta |
| 4 | MB + MF (weighted) | beta, w |
| 5 | MB separate beta per stimulus | beta_f, beta_t |
| 6 | MB separate beta per stimulus + MF | beta_f, beta_t, w |
| 7 | MB separate beta per revaluation | beta_q1, beta_q2 |
| 8 | MB separate beta per reval + MF | beta_q1, beta_q2, w |
| 9 | Separate w per stimulus | beta, w_f, w_t |
| 10 | Separate w per revaluation | beta, w_q1, w_q2 |
| 11 | Separate w per stimulus × reval | beta, w_f1, w_f2, w_t1, w_t2 |

### Cost-of-time models (joint choice + RT as deliberation cost)
RT modeled as: `log(RT) = c0 + gamma * |EV - offer| + noise`
Choice: `P(MB) = sigmoid(beta * (V - offer) - kappa * log(RT))`

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8A  MODEL DEFINITIONS FOR CBM
# ══════════════════════════════════════════════════════════════════════
# CBM models: f(parameters, data) -> log_likelihood (positive)
# Parameters in UNCONSTRAINED space (apply sigmoid/exp inside)

import pickle, os
from cbm import individual_fit, bms, hbi_main, hbi_null

CBM_DIR = f"cbm_output_{OUT_PREFIX}"
os.makedirs(CBM_DIR, exist_ok=True)

# ── Prepare per-subject data as list of dicts ──
# FIX (a): raw gamble/safe target. Filter depends on FIT_ALL_TRIALS.
if FIT_ALL_TRIALS:
    fit_df = trial_df.dropna(subset=["choice"]).copy()
else:
    fit_df = trial_df[trial_df["is_anti_mf"] == 1].dropna(subset=["choice"]).copy()
fit_df["is_feather"] = (fit_df["state"] == "Feather").astype(int)
fit_df["is_q1"] = (fit_df["query_num"] == 1).astype(int)

subject_ids = fit_df["subject_id"].unique()
N_SUBJ = len(subject_ids)

data_list = []
for sid in subject_ids:
    sdf = fit_df[fit_df["subject_id"] == sid]
    data_list.append({
        "ev":         sdf["ev"].values.astype(float),
        "mf_value":   sdf["mf_value"].values.astype(float),
        "offer":      sdf["offer"].values.astype(float),
        "choice":     sdf["choice"].values.astype(float),  # FIX (a): raw gamble/safe action (0/1), not chose_mb
        "is_feather": sdf["is_feather"].values.astype(float),
        "is_q1":      sdf["is_q1"].values.astype(float),
        "log_rt":     sdf["log_rt"].values.astype(float),
    })

print(f"CBM data prepared: {N_SUBJ} subjects, {len(fit_df)} trials ({FILTER_LABEL}; target = raw choice; dir={CBM_DIR})")

# ── Helper: Bernoulli log-likelihood ──
def _ll(p_gamble, choice):
    p = np.clip(p_gamble, 1e-8, 1 - 1e-8)
    return np.sum(choice * np.log(p) + (1 - choice) * np.log(1 - p))

# ══════════════════════════════════════════════════════════════════════
# CHOICE-ONLY MODELS (return log-likelihood)
# ══════════════════════════════════════════════════════════════════════

def m1_mf_only(params, d):
    beta = params[0]
    p = expit(beta * (d["mf_value"] - d["offer"]))
    return _ll(p, d["choice"])

def m2_mf_alpha(params, d):
    beta, alpha_raw = params[0], params[1]
    alpha = expit(alpha_raw)
    p = expit(beta * (alpha * d["mf_value"] - d["offer"]))
    return _ll(p, d["choice"])

def m3_mb_only(params, d):
    beta = params[0]
    p = expit(beta * (d["ev"] - d["offer"]))
    return _ll(p, d["choice"])

def m4_mb_mf(params, d):
    beta, w_raw = params[0], params[1]
    w = expit(w_raw)
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m5_mb_sep_stim(params, d):
    beta_f, beta_t = params[0], params[1]
    beta = np.where(d["is_feather"], beta_f, beta_t)
    p = expit(beta * (d["ev"] - d["offer"]))
    return _ll(p, d["choice"])

def m7_mb_sep_reval(params, d):
    beta_q1, beta_q2 = params[0], params[1]
    beta = np.where(d["is_q1"], beta_q1, beta_q2)
    p = expit(beta * (d["ev"] - d["offer"]))
    return _ll(p, d["choice"])

def m9_sep_w_stim(params, d):
    beta, wf_raw, wt_raw = params[0], params[1], params[2]
    w = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])
def m12_mb_mf_sep_stim(params, d):
    beta_f, beta_t,w = params[0], params[1],params[0]
    beta = np.where(d["is_feather"], beta_f, beta_t)
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m16_sep_beta_w_stim(params, d):
    """M16 — per-stimulus beta AND per-stimulus MB/MF mixing weight.
    Superset of M9 (beta shared) and M15 (w shared).
    v = w_stim * ev + (1 - w_stim) * mf_value  (= r_mid + w_stim * T in this task)
    P(MB) = sigmoid(beta_stim * (v - offer))
    """
    beta_f, beta_t, wf_raw, wt_raw = params[0], params[1], params[2], params[3]
    beta = np.where(d["is_feather"], beta_f, beta_t)
    w    = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v    = w * d["ev"] + (1 - w) * d["mf_value"]
    p    = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m10_sep_w_reval(params, d):
    beta, w1_raw, w2_raw = params[0], params[1], params[2]
    w = np.where(d["is_q1"], expit(w1_raw), expit(w2_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m11_sep_beta_w_reval(params, d):
    
    beta_f, beta_t, w = params[0], params[1], params[2]
    beta = np.where(d["is_q1"], beta_f, beta_t)
    v    = w * d["ev"] + (1 - w) * d["mf_value"]
    p    = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

def m11_sep_w_stim_reval(params, d):
    beta = params[0]
    wf1, wf2 = expit(params[1]), expit(params[2])
    wt1, wt2 = expit(params[3]), expit(params[4])
    w = np.where(d["is_feather"],
                 np.where(d["is_q1"], wf1, wf2),
                 np.where(d["is_q1"], wt1, wt2))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return _ll(p, d["choice"])

# ══════════════════════════════════════════════════════════════════════
# COST-OF-TIME MODELS (joint choice + RT)
# ══════════════════════════════════════════════════════════════════════

def _rt_ll(log_rt, ev, offer, c0, gamma, log_sigma):
    """Log-likelihood of RT component."""
    valid = ~np.isnan(log_rt)
    if valid.sum() == 0:
        return 0.0
    sigma = max(np.exp(log_sigma), 0.01)
    mu = c0 + gamma * np.abs(ev[valid] - offer[valid])
    return (-0.5 * np.sum(((log_rt[valid] - mu) / sigma)**2)
            - valid.sum() * np.log(sigma)
            - 0.5 * valid.sum() * np.log(2 * np.pi))

def _get_rt(log_rt, ev, offer, c0, gamma):
    out = log_rt.copy()
    m = np.isnan(out)
    if m.any():
        out[m] = c0 + gamma * np.abs(ev[m] - offer[m])
    return out

def ct_m3(params, d):
    beta, c0, gamma, kappa, log_sigma = params
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (d["ev"] - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

def ct_m4(params, d):
    beta, w_raw, c0, gamma, kappa, log_sigma = params
    w = expit(w_raw)
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (v - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

def ct_m9(params, d):
    beta, wf_raw, wt_raw, c0, gamma, kappa, log_sigma = params
    w = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (v - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

def ct_m11(params, d):
    beta = params[0]
    wf1, wf2, wt1, wt2 = expit(params[1]), expit(params[2]), expit(params[3]), expit(params[4])
    c0, gamma, kappa, log_sigma = params[5], params[6], params[7], params[8]
    w = np.where(d["is_feather"],
                 np.where(d["is_q1"], wf1, wf2),
                 np.where(d["is_q1"], wt1, wt2))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    rt = _get_rt(d["log_rt"], d["ev"], d["offer"], c0, gamma)
    net = beta * (v - d["offer"]) - kappa * rt
    return _ll(expit(net), d["choice"]) + _rt_ll(d["log_rt"], d["ev"], d["offer"], c0, gamma, log_sigma)

# ══════════════════════════════════════════════════════════════════════
# MODEL REGISTRY
# ══════════════════════════════════════════════════════════════════════

MODELS = {
    "M1_MF":            {"func": m1_mf_only,        "k": 1, "names": ["beta"]},
    "M2_MF_alpha":      {"func": m2_mf_alpha,       "k": 2, "names": ["beta","alpha_raw"]},
    "M3_MB":            {"func": m3_mb_only,        "k": 1, "names": ["beta"]},
    "M4_MB_MF":         {"func": m4_mb_mf,          "k": 2, "names": ["beta","w_raw"]},
    "M5_MB_stim":       {"func": m5_mb_sep_stim,    "k": 2, "names": ["beta_f","beta_t"]},
    "M7_MB_reval":      {"func": m7_mb_sep_reval,   "k": 2, "names": ["beta_q1","beta_q2"]},
    "M9_w_stim":        {"func": m9_sep_w_stim,     "k": 3, "names": ["beta","w_f_raw","w_t_raw"]},
    "M10_w_reval":      {"func": m10_sep_w_reval,   "k": 3, "names": ["beta","w_q1_raw","w_q2_raw"]},
    "m12_mb_mf_sep_stim":{"func": m12_mb_mf_sep_stim,   "k": 3, "names":["beta_f","beta_t",'w']},
    "m11_sep_beta_w_reval":{"func": m11_sep_beta_w_reval,   "k": 3, "names":["beta_f","beta_t",'w']},
    "m16_sep_beta_w_stim":{"func":m16_sep_beta_w_stim,"k":4,"names":["beta_f", "beta_t", "wf_raw", "wt_raw"]}
}

CT_MODELS = {
    "CT_M3":            {"func": ct_m3,  "k": 5, "names": ["beta","c0","gamma","kappa","log_sigma"]},
    # "CT_M4":            {"func": ct_m4,  "k": 6, "names": ["beta","w_raw","c0","gamma","kappa","log_sigma"]}
    "CT_M9":            {"func": ct_m9,  "k": 7, "names": ["beta","w_f","w_t","c0","gamma","kappa","log_sigma"]},
    # "CT_M11":           {"func": ct_m11, "k": 9, "names": ["beta","wf1","wf2","wt1","wt2","c0","gamma","kappa","log_sigma"]},
}






In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8B  CBM INDIVIDUAL FITS (MAP via Laplace approximation)
# ══════════════════════════════════════════════════════════════════════

PRIOR_VARIANCE = 6.25  # default prior variance (sd=2.5)

cbm_fits = {}

# ── Choice-only models ──
print("=" * 70)
print("FITTING CHOICE-ONLY MODELS (CBM individual_fit)")
print("=" * 70)

for mname, mspec in MODELS.items():
    k = mspec["k"]
    prior_mean = np.zeros(k)
    fname = os.path.join(CBM_DIR, f"{mname}.pkl")
    print(f"\nFitting {mname} (k={k})...")
    cbm_result = individual_fit(
        data_list, mspec["func"], prior_mean, PRIOR_VARIANCE,
        fname=fname, config={"verbose": False}
    )
    cbm_fits[mname] = cbm_result
    lme = cbm_result.output.log_evidence
    params = cbm_result.output.parameters
    print(f"  Mean log-evidence = {lme.mean():.2f}")
    for pi, pn in enumerate(mspec["names"]):
        vals = params[:, pi]
        print(f"  {pn:15s}: M={vals.mean():.3f}, SD={vals.std():.3f}")

# ── Cost-of-time models: FAST MLE (replaces slow CBM hierarchical) ──
print("\n" + "=" * 70)
print("FITTING COST-OF-TIME MODELS (Fast MLE)")
print("=" * 70)

# Filter data_list to subjects with sufficient RT
ct_data_list = []
ct_subject_mask = []
for i, d in enumerate(data_list):
    valid_rt = np.sum(~np.isnan(d["log_rt"]))
    if valid_rt >= 5:
        ct_data_list.append(d)
        ct_subject_mask.append(i)
ct_subject_mask = np.array(ct_subject_mask)
print(f"Cost-of-time: {len(ct_data_list)} subjects with >= 5 valid RT trials")

import time as _time

def _fit_ct_mle(ct_func, data_dict, k, param_names, n_starts=10):
    """Fast MLE fit for a single subject's cost-of-time model."""
    best_nll, best_params = np.inf, None
    c0_idx = param_names.index("c0") if "c0" in param_names else None
    for _ in range(n_starts):
        x0 = np.random.randn(k) * 0.5
        if c0_idx is not None:
            x0[c0_idx] = 8.0 + np.random.randn() * 0.3
        try:
            res = minimize(lambda p, d: -ct_func(p, d), x0, args=(data_dict,),
                          method='L-BFGS-B', options={"maxiter": 200})
            if res.fun < best_nll:
                best_nll, best_params = res.fun, res.x
        except Exception:
            continue
    return best_params, best_nll

ct_mle_results = {}
for mname, mspec in CT_MODELS.items():
    k = mspec["k"]
    pnames = mspec["names"]
    print(f"\nFitting {mname} (k={k}) via MLE...")
    t0 = _time.time()
    subj_params = np.zeros((len(ct_data_list), k))
    subj_nll = np.zeros(len(ct_data_list))
    for si, d in enumerate(ct_data_list):
        params, nll = _fit_ct_mle(mspec["func"], d, k, pnames)
        if params is not None:
            subj_params[si] = params
            subj_nll[si] = nll
        else:
            subj_nll[si] = np.inf
    n_trials_per = np.array([len(d["choice"]) for d in ct_data_list])
    subj_bic = k * np.log(n_trials_per) + 2 * subj_nll
    subj_aic = 2 * k + 2 * subj_nll
    ct_mle_results[mname] = {
        "params": subj_params,
        "nll": subj_nll,
        "bic": subj_bic,
        "aic": subj_aic,
        "k": k,
        "names": pnames,
    }
    elapsed = _time.time() - t0
    print(f"  Mean NLL = {subj_nll[subj_nll < np.inf].mean():.2f}  ({elapsed:.1f}s)")

print("\nAll CT models fitted via MLE.")

In [ ]:
## ══════════════════════════════════════════════════════════════════════
# 8C  BAYESIAN MODEL SELECTION
# ══════════════════════════════════════════════════════════════════════

# ── Choice-only BMS ──
model_names_choice = list(MODELS.keys())
lme_matrix = np.column_stack([cbm_fits[m].output.log_evidence for m in model_names_choice])

bms_result = bms(lme_matrix)

print("=" * 70)
print("BAYESIAN MODEL SELECTION: Choice-Only Models")
print("=" * 70)
print(f"{"Model":25s} {"Freq":>8s} {"ExcP":>8s} {"PXP":>8s} {"Mean LME":>10s}")
print("-" * 63)
for i, mname in enumerate(model_names_choice):
    print(f"{mname:25s} {bms_result.model_frequency[i]:8.3f} "
          f"{bms_result.exceedance_prob[i]:8.3f} "
          f"{bms_result.protected_exceedance_prob[i]:8.3f} "
          f"{lme_matrix[:, i].mean():10.2f}")
print(f"\nBayes Omnibus Risk: {bms_result.bor:.4f}")

best_choice_model = model_names_choice[np.argmax(bms_result.protected_exceedance_prob)]
print(f"Best choice-only model (highest PXP): {best_choice_model}")

# ── Cost-of-time: BIC-based model comparison (from MLE fits) ──
ct_model_names = list(CT_MODELS.keys())
print("\n" + "=" * 70)
print("COST-OF-TIME MODEL COMPARISON (MLE + BIC)")
print("=" * 70)
print(f"{'Model':25s} {'Mean BIC':>10s} {'Mean AIC':>10s} {'Mean NLL':>10s}")
print("-" * 58)
ct_bic_matrix = np.column_stack([ct_mle_results[m]["bic"] for m in ct_model_names])
ct_aic_matrix = np.column_stack([ct_mle_results[m]["aic"] for m in ct_model_names])
for i, mname in enumerate(ct_model_names):
    r = ct_mle_results[mname]
    valid = r["nll"] < np.inf
    print(f"{mname:25s} {r['bic'][valid].mean():10.2f} {r['aic'][valid].mean():10.2f} {r['nll'][valid].mean():10.2f}")

# Per-subject best CT model (by BIC)
ct_best_idx = np.argmin(ct_bic_matrix, axis=1)
ct_best_names = [ct_model_names[i] for i in ct_best_idx]
best_ct_model = max(set(ct_best_names), key=ct_best_names.count)
print(f"\nBest CT model (most frequent BIC winner): {best_ct_model}")
from collections import Counter as _Counter
for mname, cnt in _Counter(ct_best_names).most_common():
    print(f"  {mname:25s}: {cnt} subjects ({100*cnt/len(ct_best_names):.1f}%)")

# ── Bar plot: PXP ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(range(len(model_names_choice)), bms_result.protected_exceedance_prob,
         color=["C2" if m == best_choice_model else "C0" for m in model_names_choice], alpha=0.8)
ax1.set_yticks(range(len(model_names_choice)))
ax1.set_yticklabels(model_names_choice, fontsize=9)
ax1.set_xlabel("Protected Exceedance Probability")
ax1.set_title("Choice-Only Models", fontweight="bold")
ax1.invert_yaxis()
ax1.axvline(1/len(model_names_choice), color="grey", ls="--", alpha=0.5, label="chance")
ax1.legend(fontsize=8)

# CT model comparison: show proportion of subjects where each model wins
ct_win_counts = np.array([ct_best_names.count(m) for m in ct_model_names])
ct_win_freq = ct_win_counts / ct_win_counts.sum()
ax2.barh(range(len(ct_model_names)), ct_win_freq,
         color=["C2" if m == best_ct_model else "C1" for m in ct_model_names], alpha=0.8)
ax2.set_yticks(range(len(ct_model_names)))
ax2.set_yticklabels(ct_model_names, fontsize=9)
ax2.set_xlabel("Proportion Best (BIC)")
ax2.set_title("Cost-of-Time Models (MLE)", fontweight="bold")
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig(f"fig_bms_pxp_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8D  HIERARCHICAL BAYESIAN INFERENCE (HBI)
# ══════════════════════════════════════════════════════════════════════
# HBI jointly refines parameters and model assignments across subjects

# ── Choice-only HBI ──
choice_model_funcs = [MODELS[m]["func"] for m in model_names_choice]
choice_cbm_paths   = [os.path.join(CBM_DIR, f"{m}.pkl") for m in model_names_choice]

print("Running HBI on choice-only models...")
hbi_result = hbi_main(
    data_list, choice_model_funcs, choice_cbm_paths,
    fname=os.path.join(CBM_DIR, "hbi_choice.pkl"),
    config={"verbose": 1, "save_prog": False}
)

# Protected exceedance
hbi_result = hbi_null(data_list, os.path.join(CBM_DIR, "hbi_choice.pkl"))

print("\n" + "=" * 70)
print("HBI RESULTS: Choice-Only Models")
print("=" * 70)
print(f"{"Model":25s} {"Freq":>8s} {"PXP":>8s}")
for i, mname in enumerate(model_names_choice):
    print(f"{mname:25s} {hbi_result.output.model_frequency[i]:8.3f} "
          f"{hbi_result.output.protected_exceedance_prob[i]:8.3f}")

# ── Per-subject winning model ──
responsibility = hbi_result.output.responsibility  # N x K
winning_model_idx = np.argmax(responsibility, axis=1)
winning_model_names = [model_names_choice[i] for i in winning_model_idx]

print(f"\nPer-subject winning model distribution:")
from collections import Counter
for mname, count in Counter(winning_model_names).most_common():
    print(f"  {mname:25s}: {count} subjects ({100*count/N_SUBJ:.1f}%)")

# ── Store per-subject winning params ──
# HBI returns refined parameters per model: output.parameters[k] is (N, d_k)
winning_params = {}
for i, sid in enumerate(subject_ids):
    k = winning_model_idx[i]
    mname = model_names_choice[k]
    params = hbi_result.output.parameters[k][i, :]
    winning_params[sid] = {
        "model": mname,
        "params": params,
        "param_names": MODELS[mname]["names"],
        "responsibility": responsibility[i, :],
    }

print(f"\nStored winning model + params for {len(winning_params)} subjects")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8E  POSTERIOR PREDICTIVE CHECKS
# ══════════════════════════════════════════════════════════════════════
# Simulate choices (and RT for CT models) from each subject's winning
# model + HBI-refined params, then compare model-agnostic patterns
# between real and simulated data.

N_SIM = 50  # simulations per subject
np.random.seed(42)

# ── Simulate from winning choice-only models ──
sim_rows = []
for subj_i, sid in enumerate(subject_ids):
    d = data_list[subj_i]
    info = winning_params[sid]
    mname = info["model"]
    params = info["params"]
    func = MODELS[mname]["func"]
    n_trials = len(d["choice"])

    # Get P(chose_MB) per trial by computing model predictions
    # For each trial, compute log-lik at choice=1 and choice=0
    p_mb = np.zeros(n_trials)
    for t in range(n_trials):
        d1 = {k: np.array([v[t]]) for k, v in d.items()}
        d1["choice"] = np.array([1.0])
        ll1 = func(params, d1)
        d1["choice"] = np.array([0.0])
        ll0 = func(params, d1)
        # ll1 = log(p), ll0 = log(1-p) for Bernoulli
        p_mb[t] = np.exp(ll1)

    for sim in range(N_SIM):
        sim_choices = (np.random.rand(n_trials) < p_mb).astype(float)
        for t in range(n_trials):
            sim_rows.append({
                "subject_id": sid,
                "sim": sim,
                "ev": d["ev"][t],
                "mf_value": d["mf_value"][t],
                "offer": d["offer"][t],
                "is_feather": d["is_feather"][t],
                "is_q1": d["is_q1"][t],
                "abs_ev_diff": np.abs(d["ev"][t] - d["offer"][t]),
                "chose_mb_sim": sim_choices[t],
                "chose_mb_real": d["choice"][t],
                "log_rt_real": d["log_rt"][t],
                "somatic_anxiety": fit_df[fit_df["subject_id"]==sid]["somatic_anxiety"].iloc[0],
            })

sim_df = pd.DataFrame(sim_rows)
median_diff = sim_df["abs_ev_diff"].median()
sim_df["difficulty"] = np.where(sim_df["abs_ev_diff"] <= median_diff, "Hard", "Easy")
median_anx = sim_df.groupby("subject_id")["somatic_anxiety"].first().median()
sim_df["anx_group"] = sim_df["subject_id"].map(
    sim_df.groupby("subject_id")["somatic_anxiety"].first().apply(
        lambda a: "High" if a >= median_anx else "Low"))

print(f"Posterior predictive simulations: {len(sim_df)} rows ({N_SIM} sims x {len(fit_df)} trials)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8F  POSTERIOR PREDICTIVE CHECK PLOTS
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = "#FAFBFC"

# ── Helper: compute pattern for real vs simulated data ──
def pattern_by_group(df, groupcols, real_col, sim_col):
    """Compute mean real and mean simulated across group levels."""
    real = df.groupby(groupcols)[real_col].mean().reset_index()
    real.columns = list(groupcols) + ["real"]
    sim = df.groupby(groupcols + ["sim"])[sim_col].mean().reset_index()
    sim_agg = sim.groupby(groupcols)[sim_col].agg(["mean","std"]).reset_index()
    sim_agg.columns = list(groupcols) + ["sim_mean","sim_std"]
    return real.merge(sim_agg, on=list(groupcols))

# ══════════════════════════════════════════════════════════════════════
# PPC 1: P(chose MB) by Difficulty
# ══════════════════════════════════════════════════════════════════════
ppc1 = pattern_by_group(sim_df, ["difficulty"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 2: P(chose MB) by Stimulus (feather vs triangle)
# ══════════════════════════════════════════════════════════════════════
sim_df["stimulus"] = np.where(sim_df["is_feather"]==1, "Feather", "Triangle")
ppc2 = pattern_by_group(sim_df, ["stimulus"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 3: P(chose MB) by Query (reval 1 vs 2)
# ══════════════════════════════════════════════════════════════════════
sim_df["query"] = np.where(sim_df["is_q1"]==1, "Query 1", "Query 2")
ppc3 = pattern_by_group(sim_df, ["query"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 4: P(chose MB) by Difficulty × Somatic Anxiety (the key 2x2)
# ══════════════════════════════════════════════════════════════════════
ppc4 = pattern_by_group(sim_df, ["difficulty","anx_group"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PPC 5: P(chose MB) by Difficulty × Stimulus
# ══════════════════════════════════════════════════════════════════════
ppc5 = pattern_by_group(sim_df, ["difficulty","stimulus"], "chose_mb_real", "chose_mb_sim")

# ══════════════════════════════════════════════════════════════════════
# PLOT: 2×3 grid of posterior predictive checks
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=BG_COLOR)

def ppc_bar(ax, ppc_df, group_col, title):
    """Bar plot: real (solid) vs simulated (hatched) for each group level."""
    levels = ppc_df[group_col].values
    x = np.arange(len(levels))
    w = 0.35
    ax.bar(x - w/2, ppc_df["real"], w, color="C0", alpha=0.8, label="Real data")
    ax.bar(x + w/2, ppc_df["sim_mean"], w, color="C3", alpha=0.6,
           yerr=ppc_df["sim_std"], capsize=4, label="Simulated (model)")
    ax.set_xticks(x)
    ax.set_xticklabels(levels)
    ax.set_ylabel("P(Chose MB)")
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color="grey", ls="--", alpha=0.4)
    ax.set_facecolor(BG_COLOR)

# Row 1: simple patterns
ppc_bar(axes[0,0], ppc1, "difficulty", "PPC 1: Difficulty")
ppc_bar(axes[0,1], ppc2, "stimulus", "PPC 2: Stimulus")
ppc_bar(axes[0,2], ppc3, "query", "PPC 3: Query")

# Row 2: interaction patterns
# PPC 4: Difficulty × Somatic Anxiety
ax = axes[1,0]; ax.set_facecolor(BG_COLOR)
for anx_i, anx in enumerate(["Low", "High"]):
    sub = ppc4[ppc4["anx_group"] == anx]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + anx_i * 0.2
    color_r = "C0" if anx == "Low" else "C3"
    color_s = "C4" if anx == "Low" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{anx} ANX (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{anx} ANX (sim)")
    ax.errorbar([x+offset for x in xs], sub["sim_mean"],
               yerr=sub["sim_std"], fmt="none", color=color_s, capsize=3)
    # Connect with lines
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_ylabel("P(Chose MB)")
ax.set_title("PPC 4: Difficulty x Somatic Anxiety", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)

# PPC 5: Difficulty × Stimulus
ax = axes[1,1]; ax.set_facecolor(BG_COLOR)
for stim_i, stim in enumerate(["Feather", "Triangle"]):
    sub = ppc5[ppc5["stimulus"] == stim]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + stim_i * 0.2
    color_r = "C0" if stim == "Feather" else "C3"
    color_s = "C4" if stim == "Feather" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{stim} (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{stim} (sim)")
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_title("PPC 5: Difficulty x Stimulus", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)

# PPC 6: Subject-level scatter — real vs simulated MB rate
ax = axes[1,2]; ax.set_facecolor(BG_COLOR)
subj_real = sim_df.groupby("subject_id")["chose_mb_real"].mean()
subj_sim  = sim_df.groupby(["subject_id","sim"])["chose_mb_sim"].mean().groupby("subject_id").mean()
merged = pd.DataFrame({"real": subj_real, "sim": subj_sim}).dropna()
ax.scatter(merged["real"], merged["sim"], alpha=0.5, s=25)
ax.plot([0,1],[0,1], "k--", alpha=0.5, lw=1)
r_ppc, p_ppc = stats.pearsonr(merged["real"], merged["sim"])
ax.set_xlabel("Real P(chose MB)")
ax.set_ylabel("Simulated P(chose MB)")
ax.set_title(f"PPC 6: Subject-level (r={r_ppc:.3f})", fontweight="bold", fontsize=10)

fig.suptitle("Posterior Predictive Checks: Real vs Model-Simulated Data",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_ppc_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# Print numerical summary
print("Posterior Predictive Check Summary:")
for name, ppc_data, gcol in [("Difficulty", ppc1, "difficulty"),
                               ("Stimulus", ppc2, "stimulus"),
                               ("Query", ppc3, "query")]:
    print(f"\n  {name}:")
    for _, row in ppc_data.iterrows():
        print(f"    {row[gcol]:>10s}: real={row['real']:.3f}  sim={row['sim_mean']:.3f} +/- {row['sim_std']:.3f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8F-ii  RT POSTERIOR PREDICTIVE CHECK (from MLE cost-of-time models)
# ══════════════════════════════════════════════════════════════════════
# Simulate RTs from the best-fit CT model for each subject, then
# compare simulated RT patterns to real RT patterns.

BG_COLOR = "#FAFBFC"
N_RT_SIM = 100
np.random.seed(42)

# ── Get best CT model params per subject ──
rt_sim_rows = []
for si, d in enumerate(ct_data_list):
    subj_idx = ct_subject_mask[si]
    sid = subject_ids[subj_idx]
    best_mi = ct_best_idx[si]
    best_mname = ct_model_names[best_mi]
    best_params = ct_mle_results[best_mname]["params"][si]
    mspec = CT_MODELS[best_mname]
    pnames = mspec["names"]

    # Extract RT params
    c0 = best_params[pnames.index("c0")]
    gamma = best_params[pnames.index("gamma")]
    log_sigma = best_params[pnames.index("log_sigma")]
    sigma_rt = max(np.exp(log_sigma), 0.01)

    # Get per-trial EV (use MB EV or blended depending on model)
    if "w_raw" in pnames:
        w_idx = [i for i, n in enumerate(pnames) if n in ("w_raw", "w_f", "w_t")]
        if "w_f" in pnames and "w_t" in pnames:
            wf = expit(best_params[pnames.index("w_f")])
            wt = expit(best_params[pnames.index("w_t")])
            w_arr = np.where(d["is_feather"], wf, wt)
        elif "w_raw" in pnames:
            w_arr = expit(best_params[pnames.index("w_raw")])
        else:
            w_arr = 1.0
        ev_eff = w_arr * d["ev"] + (1 - w_arr) * d["mf_value"]
    else:
        ev_eff = d["ev"]

    n_trials = len(d["choice"])
    valid_rt = ~np.isnan(d["log_rt"])

    for sim in range(N_RT_SIM):
        # Simulate log(RT) from the model
        mu_rt = c0 + gamma * np.abs(ev_eff - d["offer"])
        sim_log_rt = mu_rt + sigma_rt * np.random.randn(n_trials)

        for t in range(n_trials):
            if not valid_rt[t]:
                continue
            rt_sim_rows.append({
                "subject_id": sid,
                "sim": sim,
                "ev": d["ev"][t],
                "mf_value": d["mf_value"][t],
                "offer": d["offer"][t],
                "is_feather": d["is_feather"][t],
                "is_q1": d["is_q1"][t],
                "abs_ev_diff": np.abs(d["ev"][t] - d["offer"][t]),
                "log_rt_real": d["log_rt"][t],
                "log_rt_sim": sim_log_rt[t],
                "somatic_anxiety": fit_df[fit_df["subject_id"]==sid]["somatic_anxiety"].iloc[0],
            })

rt_sim_df = pd.DataFrame(rt_sim_rows)
median_diff = rt_sim_df["abs_ev_diff"].median()
rt_sim_df["difficulty"] = np.where(rt_sim_df["abs_ev_diff"] <= median_diff, "Hard", "Easy")
median_anx = rt_sim_df.groupby("subject_id")["somatic_anxiety"].first().median()
rt_sim_df["anx_group"] = rt_sim_df["subject_id"].map(
    rt_sim_df.groupby("subject_id")["somatic_anxiety"].first().apply(
        lambda a: "High" if a >= median_anx else "Low"))
rt_sim_df["stimulus"] = np.where(rt_sim_df["is_feather"]==1, "Feather", "Triangle")
rt_sim_df["query"] = np.where(rt_sim_df["is_q1"]==1, "Query 1", "Query 2")

# Convert log RT to RT in ms for interpretability
rt_sim_df["rt_real_ms"] = np.exp(rt_sim_df["log_rt_real"])
rt_sim_df["rt_sim_ms"] = np.exp(rt_sim_df["log_rt_sim"])

print(f"RT PPC: {len(rt_sim_df)} rows ({N_RT_SIM} sims x trials with valid RT)")

# ══════════════════════════════════════════════════════════════════════
# RT PPC PLOTS
# ══════════════════════════════════════════════════════════════════════

def rt_pattern(df, groupcols):
    """Compute mean real and simulated RT across group levels."""
    real = df.groupby(groupcols)["rt_real_ms"].mean().reset_index()
    real.columns = list(groupcols) + ["real"]
    sim = df.groupby(groupcols + ["sim"])["rt_sim_ms"].mean().reset_index()
    sim_agg = sim.groupby(groupcols)["rt_sim_ms"].agg(["mean","std"]).reset_index()
    sim_agg.columns = list(groupcols) + ["sim_mean","sim_std"]
    return real.merge(sim_agg, on=list(groupcols))

def rt_ppc_bar(ax, ppc_df, group_col, title):
    """Bar plot: real vs simulated RT."""
    levels = ppc_df[group_col].values
    x = np.arange(len(levels))
    w = 0.35
    ax.bar(x - w/2, ppc_df["real"], w, color="C0", alpha=0.8, label="Real RT")
    ax.bar(x + w/2, ppc_df["sim_mean"], w, color="C3", alpha=0.6,
           yerr=ppc_df["sim_std"], capsize=4, label="Simulated RT")
    ax.set_xticks(x)
    ax.set_xticklabels(levels)
    ax.set_ylabel("Mean RT (ms)")
    ax.set_title(title, fontweight="bold", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_facecolor(BG_COLOR)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor=BG_COLOR)

# PPC-RT1: RT by difficulty
ppc_rt1 = rt_pattern(rt_sim_df, ["difficulty"])
rt_ppc_bar(axes[0,0], ppc_rt1, "difficulty", "RT-PPC 1: Difficulty")

# PPC-RT2: RT by stimulus
ppc_rt2 = rt_pattern(rt_sim_df, ["stimulus"])
rt_ppc_bar(axes[0,1], ppc_rt2, "stimulus", "RT-PPC 2: Stimulus")

# PPC-RT3: RT by query
ppc_rt3 = rt_pattern(rt_sim_df, ["query"])
rt_ppc_bar(axes[0,2], ppc_rt3, "query", "RT-PPC 3: Query")

# PPC-RT4: Difficulty x Somatic Anxiety (key 2x2)
ax = axes[1,0]; ax.set_facecolor(BG_COLOR)
ppc_rt4 = rt_pattern(rt_sim_df, ["difficulty", "anx_group"])
for anx_i, anx in enumerate(["Low", "High"]):
    sub = ppc_rt4[ppc_rt4["anx_group"] == anx]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + anx_i * 0.2
    color_r = "C0" if anx == "Low" else "C3"
    color_s = "C4" if anx == "Low" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{anx} ANX (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{anx} ANX (sim)")
    ax.errorbar([x+offset for x in xs], sub["sim_mean"],
               yerr=sub["sim_std"], fmt="none", color=color_s, capsize=3)
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_ylabel("Mean RT (ms)")
ax.set_title("RT-PPC 4: Difficulty x Somatic Anxiety", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)

# PPC-RT5: Difficulty x Stimulus
ax = axes[1,1]; ax.set_facecolor(BG_COLOR)
ppc_rt5 = rt_pattern(rt_sim_df, ["difficulty", "stimulus"])
for stim_i, stim in enumerate(["Feather", "Triangle"]):
    sub = ppc_rt5[ppc_rt5["stimulus"] == stim]
    x_order = {"Hard": 0, "Easy": 1}
    xs = [x_order[d] for d in sub["difficulty"]]
    offset = -0.1 + stim_i * 0.2
    color_r = "C0" if stim == "Feather" else "C3"
    color_s = "C4" if stim == "Feather" else "C1"
    ax.scatter([x+offset for x in xs], sub["real"], marker="o", s=80,
              color=color_r, zorder=5, label=f"{stim} (real)")
    ax.scatter([x+offset for x in xs], sub["sim_mean"], marker="s", s=60,
              color=color_s, zorder=5, label=f"{stim} (sim)")
    ax.plot([x+offset for x in xs], sub["real"].values, color=color_r, ls="-", lw=1.5)
    ax.plot([x+offset for x in xs], sub["sim_mean"].values, color=color_s, ls="--", lw=1.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Hard", "Easy"])
ax.set_title("RT-PPC 5: Difficulty x Stimulus", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)

# PPC-RT6: Subject-level scatter (mean log RT)
ax = axes[1,2]; ax.set_facecolor(BG_COLOR)
subj_rt_real = rt_sim_df.groupby("subject_id")["log_rt_real"].mean()
subj_rt_sim = rt_sim_df.groupby(["subject_id","sim"])["log_rt_sim"].mean().groupby("subject_id").mean()
merged_rt = pd.DataFrame({"real": subj_rt_real, "sim": subj_rt_sim}).dropna()
ax.scatter(merged_rt["real"], merged_rt["sim"], alpha=0.5, s=25)
lims_rt = [min(merged_rt.min()), max(merged_rt.max())]
ax.plot(lims_rt, lims_rt, "k--", alpha=0.5, lw=1)
r_rt, p_rt = stats.pearsonr(merged_rt["real"], merged_rt["sim"])
ax.set_xlabel("Real mean log(RT)")
ax.set_ylabel("Simulated mean log(RT)")
ax.set_title(f"RT-PPC 6: Subject-level (r={r_rt:.3f})", fontweight="bold", fontsize=10)

fig.suptitle("RT Posterior Predictive Checks: Real vs Model-Simulated Reaction Times",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_rt_ppc_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ── Numerical summary ──
print("RT Posterior Predictive Check Summary:")
for name, ppc_data, gcol in [("Difficulty", ppc_rt1, "difficulty"),
                               ("Stimulus", ppc_rt2, "stimulus"),
                               ("Query", ppc_rt3, "query")]:
    print(f"\n  {name}:")
    for _, row in ppc_data.iterrows():
        print(f"    {row[gcol]:>10s}: real={row['real']:.0f}ms  sim={row['sim_mean']:.0f}ms +/- {row['sim_std']:.0f}ms")

print(f"\n  Subject-level log(RT) correlation: r = {r_rt:.3f}, p = {p_rt:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8D-ii  MODEL RESPONSIBILITY & FREQUENCY PLOT
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = "#FAFBFC"

# responsibility: N_subjects x K_models (from HBI)
# Each row sums to 1 — posterior P(model k | subject i)

fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor=BG_COLOR,
                          gridspec_kw={"width_ratios": [1.2, 2, 1]})

# ── Panel A: Winning model frequency (bar chart) ──
ax = axes[0]; ax.set_facecolor(BG_COLOR)
from collections import Counter
win_counts = Counter(winning_model_names)
# Sort by model number
sorted_models = sorted(win_counts.keys(), key=lambda m: list(MODELS.keys()).index(m))
counts = [win_counts[m] for m in sorted_models]
colors = ["C2" if m == best_choice_model else "C0" for m in sorted_models]
ax.barh(range(len(sorted_models)), counts, color=colors, alpha=0.85)
ax.set_yticks(range(len(sorted_models)))
ax.set_yticklabels(sorted_models, fontsize=9)
ax.set_xlabel("N subjects (best model)")
ax.set_title("Winning Model Frequency", fontweight="bold", fontsize=11)
ax.invert_yaxis()
for i, v in enumerate(counts):
    if v > 0:
        ax.text(v + 0.5, i, str(v), va="center", fontsize=9)

# ── Panel B: Full responsibility matrix (heatmap) ──
ax = axes[1]; ax.set_facecolor(BG_COLOR)
# Sort subjects by winning model, then by confidence
sort_idx = np.lexsort((responsibility.max(axis=1),  # secondary: confidence
                        winning_model_idx))          # primary: winning model
resp_sorted = responsibility[sort_idx, :]
im = ax.imshow(resp_sorted.T, aspect="auto", cmap="Blues", vmin=0, vmax=1,
               interpolation="nearest")
ax.set_yticks(range(len(model_names_choice)))
ax.set_yticklabels(model_names_choice, fontsize=8)
ax.set_xlabel(f"Subjects (n={N_SUBJ}, sorted by winning model)")
ax.set_title("Model Responsibility Matrix\nP(model | subject)", fontweight="bold", fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8, label="Responsibility")

# ── Panel C: Confidence distribution ──
ax = axes[2]; ax.set_facecolor(BG_COLOR)
max_resp = responsibility.max(axis=1)
ax.hist(max_resp, bins=20, color="C0", alpha=0.7, edgecolor="white")
ax.axvline(1/len(model_names_choice), color="grey", ls="--", alpha=0.6,
           label=f"Chance ({1/len(model_names_choice):.2f})")
ax.axvline(max_resp.mean(), color="C3", ls="-", lw=2,
           label=f"Mean ({max_resp.mean():.2f})")
ax.set_xlabel("Max responsibility (confidence)")
ax.set_ylabel("N subjects")
ax.set_title("Assignment Confidence", fontweight="bold", fontsize=11)
ax.legend(fontsize=8)

fig.suptitle("HBI Model Assignment: Per-Subject Responsibility",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_model_responsibility_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

print(f"Mean max responsibility: {max_resp.mean():.3f} (SD={max_resp.std():.3f})")
print(f"Subjects with confidence > 0.8: {(max_resp > 0.8).sum()} / {N_SUBJ}")
print(f"Subjects with confidence > 0.5: {(max_resp > 0.5).sum()} / {N_SUBJ}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# GROUP COLLAPSE: MF (M2) vs MB (M3 + M7 + M9)
# ══════════════════════════════════════════════════════════════════════
# After PPC, collapse winning models into two cognitive groups for all
# downstream analyses:
#   MF — M2_MF_alpha  (pure model-free)
#   MB — M7_MB_reval + M9_w_stim + M3_MB  (all predominantly MB)
# Each subject's winning_params[sid]["model"] is overwritten to "MF" or
# "MB"; the original name is preserved in winning_params[sid]["original_model"].

GROUP_MAP = {
    # --- Model-free: choice driven only by cached stage value (no EV/tree term) ---
    "M1_MF":                "MF",
    "M2_MF_alpha":          "MF",
    # --- Model-based: EV (tree) term enters the choice ---
    "M3_MB":                "MB",
    "M4_MB_MF":             "MB",
    "M5_MB_stim":           "MB",
    "M7_MB_reval":          "MB",
    "M9_w_stim":            "MB",
    "M10_w_reval":          "MB",
    "m11_sep_beta_w_reval": "MB",
    "m12_mb_mf_sep_stim":   "MB",
    "m16_sep_beta_w_stim":  "MB",
}

for sid, info in winning_params.items():
    info.setdefault("original_model", info["model"])
    info["model"] = GROUP_MAP.get(info["original_model"], info["original_model"])

from collections import Counter
print("Post-collapse group assignments:")
for grp, cnt in Counter(v["model"] for v in winning_params.values()).most_common():
    print(f"  {grp:25s}: {cnt} subjects")

# Canonical constants for all downstream cells
MODEL_GROUPS = ["MF", "MB"]
MODEL_COLORS = {"MF": "#3498DB", "MB": "#2ECC71"}
MODEL_LABELS = {"MF": "MF-only (M2)",
                "MB": "Model-Based (M3 + M7 + M9)"}
MODEL_SHORT  = {"MF": "MF", "MB": "MB"}


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 8K  P(Chose MB) BY STIMULUS: MF vs MB Groups
# ══════════════════════════════════════════════════════════════════════

BG_COLOR = "#FAFBFC"

# Two-group palette + labels (MF vs MB collapsed groups)
MODEL_GROUPS_329 = ["MF", "MB"]
MODEL_COLORS_329 = {"MF": "#3498DB", "MB": "#2ECC71"}  # blue vs green
MODEL_LABELS_329 = {
    "MF": "MF-only\n(M2)",
    "MB": "Model-Based\n(M3 + M7 + M9)",
}

# Tag each trial with the subject's winning model
model_map = {sid: info["model"] for sid, info in winning_params.items()}
plot_df = fit_df.copy()
plot_df["winning_model"] = plot_df["subject_id"].map(model_map)

# Filter to the three groups
m3 = plot_df[plot_df["winning_model"].isin(MODEL_GROUPS_329)].copy()
m3["stimulus"] = np.where(m3["state"] == "Feather", "Feather", "Triangle")
m3["model_label"] = m3["winning_model"].map(MODEL_LABELS_329)

n_by_model = {m: m3[m3["winning_model"] == m]["subject_id"].nunique() for m in MODEL_GROUPS_329}
for m in MODEL_GROUPS_329:
    print(f"{MODEL_LABELS_329[m].split(chr(10))[0]}: {n_by_model[m]} subjects")

# ── Compute cell means ──
cell_means = m3.groupby(["winning_model", "stimulus"]).agg(
    mb_mean=("chose_mb", "mean"),
    mb_sem=("chose_mb", "sem"),
    n=("chose_mb", "count"),
).reset_index()

cell_means_q = m3.groupby(["winning_model", "stimulus", "query_num"]).agg(
    mb_mean=("chose_mb", "mean"),
    mb_sem=("chose_mb", "sem"),
    n=("chose_mb", "count"),
).reset_index()

# ── Plot: 2 panels ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG_COLOR)

# Panel A: P(chose MB) by stimulus, MF / MB
ax = ax1; ax.set_facecolor(BG_COLOR)
stimuli = ["Feather", "Triangle"]
x = np.arange(len(stimuli))
n_models = len(MODEL_GROUPS_329)
w = 0.8 / n_models  # total group width 0.8

for i, mname in enumerate(MODEL_GROUPS_329):
    sub = cell_means[cell_means["winning_model"] == mname].set_index("stimulus").reindex(stimuli)
    offs = (i - (n_models - 1) / 2) * w
    ax.bar(x + offs, sub["mb_mean"], w, yerr=1.96 * sub["mb_sem"],
           label=MODEL_LABELS_329[mname], color=MODEL_COLORS_329[mname],
           capsize=5, alpha=0.85, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(["Feather\n(pos->neg reval)", "Triangle\n(neg->pos reval)"])
ax.set_ylabel("P(Chose MB)", fontsize=11)
ax.set_title("MB Choice Rate by Stimulus", fontweight="bold", fontsize=12)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)
ax.legend(fontsize=7, loc="lower right")
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Panel B: Split by query (reval 1 vs 2)
ax = ax2; ax.set_facecolor(BG_COLOR)
cells_b = [("Feather", 1), ("Feather", 2), ("Triangle", 1), ("Triangle", 2)]
x = np.arange(len(cells_b))
xlabels = ["Feather\nReval 1", "Feather\nReval 2", "Triangle\nReval 1", "Triangle\nReval 2"]

for i, mname in enumerate(MODEL_GROUPS_329):
    sub = cell_means_q[cell_means_q["winning_model"] == mname].copy()
    sub["key"] = list(zip(sub["stimulus"], sub["query_num"]))
    sub = sub.set_index("key").reindex(cells_b)
    offs = (i - (n_models - 1) / 2) * w
    ax.bar(x + offs, sub["mb_mean"], w, yerr=1.96 * sub["mb_sem"],
           label=MODEL_LABELS_329[mname], color=MODEL_COLORS_329[mname],
           capsize=4, alpha=0.85, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel("P(Chose MB)", fontsize=11)
ax.set_title("MB Choice Rate by Stimulus x Query", fontweight="bold", fontsize=12)
ax.axhline(0.5, color="grey", ls="--", alpha=0.4)
ax.legend(fontsize=7, loc="lower right")
ax.set_ylim(0, 1)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("MF vs MB: Choice Rates on Anti-MF Trials",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_mb_choice_mf_vs_mb_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ── Print stats ──
print("\nCell means:")
for _, row in cell_means.iterrows():
    mlbl = MODEL_LABELS_329[row["winning_model"]].split(chr(10))[0]
    print(f"  {mlbl:>30s} x {row['stimulus']:>10s}: "
          f"P(MB)={row['mb_mean']:.3f} +/- {row['mb_sem']:.3f}  (n={int(row['n'])})")

# ── Pairwise tests per stimulus ──
from itertools import combinations
print("\nPairwise Mann-Whitney (per stimulus, subject-mean P(MB)):")
for stim in stimuli:
    per_subj = {}
    for mname in MODEL_GROUPS_329:
        sub = m3[(m3["winning_model"] == mname) & (m3["stimulus"] == stim)]
        per_subj[mname] = sub.groupby("subject_id")["chose_mb"].mean()
    print(f"  [{stim}]")
    for a, b in combinations(MODEL_GROUPS_329, 2):
        u_stat, u_p = stats.mannwhitneyu(per_subj[a], per_subj[b], alternative="two-sided")
        la = MODEL_LABELS_329[a].split(chr(10))[0]
        lb = MODEL_LABELS_329[b].split(chr(10))[0]
        print(f"    {la} (M={per_subj[a].mean():.3f}) vs {lb} (M={per_subj[b].mean():.3f}): "
              f"U={u_stat:.0f}, p={u_p:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RANK-BASED MEDIATION HELPER
# ══════════════════════════════════════════════════════════════════════
# All mediations use rank-transformed variables (Conover & Iman, 1982)
# to avoid normality/linearity assumptions. Equivalent to running
# path analysis on Spearman ranks. Bootstrap CI on indirect effect.

import statsmodels.formula.api as smf
from scipy.stats import rankdata

def rank_mediation(df, x_col, m_col, y_col, n_boot=5000, seed=42, label=""):
    """Run rank-based mediation: X -> M -> Y with bootstrap CI."""
    work = df[[x_col, m_col, y_col]].dropna().copy()
    n = len(work)
    if n < 15:
        print(f"  WARNING: n={n} too small for mediation. Skipping.")
        return None

    # Rank-transform all variables
    work["X_r"] = rankdata(work[x_col])
    work["M_r"] = rankdata(work[m_col])
    work["Y_r"] = rankdata(work[y_col])

    # Standardize ranks for interpretable betas
    for col in ["X_r", "M_r", "Y_r"]:
        work[col] = (work[col] - work[col].mean()) / work[col].std()

    # Path c: X -> Y (total)
    mod_c = smf.ols("Y_r ~ X_r", data=work).fit()
    c_total = mod_c.params["X_r"]
    c_p = mod_c.pvalues["X_r"]

    # Path a: X -> M
    mod_a = smf.ols("M_r ~ X_r", data=work).fit()
    a = mod_a.params["X_r"]
    a_p = mod_a.pvalues["X_r"]

    # Path b + c': Y ~ X + M
    mod_bc = smf.ols("Y_r ~ X_r + M_r", data=work).fit()
    b = mod_bc.params["M_r"]
    b_p = mod_bc.pvalues["M_r"]
    c_prime = mod_bc.params["X_r"]
    c_prime_p = mod_bc.pvalues["X_r"]

    # Indirect effect + bootstrap
    indirect = a * b
    np.random.seed(seed)
    boot_ab = np.zeros(n_boot)
    for bi in range(n_boot):
        idx = np.random.choice(n, size=n, replace=True)
        bdf = work.iloc[idx]
        # Re-rank within bootstrap sample
        bdf = bdf.copy()
        bdf["X_r"] = rankdata(bdf[x_col])
        bdf["M_r"] = rankdata(bdf[m_col])
        bdf["Y_r"] = rankdata(bdf[y_col])
        for col in ["X_r", "M_r", "Y_r"]:
            bdf[col] = (bdf[col] - bdf[col].mean()) / bdf[col].std()
        try:
            ba = smf.ols("M_r ~ X_r", data=bdf).fit().params["X_r"]
            bb = smf.ols("Y_r ~ X_r + M_r", data=bdf).fit().params["M_r"]
            boot_ab[bi] = ba * bb
        except:
            boot_ab[bi] = np.nan

    boot_ab = boot_ab[~np.isnan(boot_ab)]
    ci_lo, ci_hi = np.percentile(boot_ab, [2.5, 97.5])
    sig = (ci_lo > 0) or (ci_hi < 0)

    # Print results
    print(f"\n  Path c (total): {x_col} -> {y_col}")
    print(f"    beta = {c_total:.4f}, p = {c_p:.4f}")
    print(f"\n  Path a: {x_col} -> {m_col}")
    print(f"    beta = {a:.4f}, p = {a_p:.4f}")
    print(f"\n  Path b: {m_col} -> {y_col} (controlling for {x_col})")
    print(f"    beta = {b:.4f}, p = {b_p:.4f}")
    print(f"\n  Path c' (direct): {x_col} -> {y_col} (controlling for {m_col})")
    print(f"    beta = {c_prime:.4f}, p = {c_prime_p:.4f}")
    print(f"\n  Indirect effect (a * b) = {indirect:.4f}")
    print(f"  Bootstrap 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  Mediation significant: {sig}")
    if abs(c_total) > 0.001:
        print(f"  Proportion mediated: {indirect / c_total:.3f}")

    # Path diagram
    print(f"\n  Path diagram:")
    print(f"                   a = {a:.3f} (p={a_p:.3f})")
    print(f"    {x_col} ────> {m_col}")
    print(f"       |                       |")
    print(f"       |                       | b = {b:.3f} (p={b_p:.3f})")
    print(f"       |                       v")
    print(f"       +──────────────> {y_col}")
    print(f"         c' = {c_prime:.3f} (p={c_prime_p:.3f})")
    print(f"         c  = {c_total:.3f} (p={c_p:.3f}) [total]")

    return {"a": a, "a_p": a_p, "b": b, "b_p": b_p,
            "c": c_total, "c_p": c_p, "c_prime": c_prime, "c_prime_p": c_prime_p,
            "indirect": indirect, "ci": (ci_lo, ci_hi), "sig": sig, "n": n,
            "work": work, "x_col": x_col, "m_col": m_col, "y_col": y_col}


def plot_mediation(res, title_str):
    """3-panel scatter for a rank-mediation result."""
    BG_COLOR = "#FAFBFC"
    work = res["work"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor=BG_COLOR)
    paths = [
        ("X_r", "M_r", f"Path a: {res["x_col"]} -> {res["m_col"]}", res["a"], res["a_p"]),
        ("M_r", "Y_r", f"Path b: {res["m_col"]} -> {res["y_col"]}", res["b"], res["b_p"]),
        ("X_r", "Y_r", f"Path c: {res["x_col"]} -> {res["y_col"]}", res["c"], res["c_p"]),
    ]
    for j, (xc, yc, ptitle, beta, pval) in enumerate(paths):
        ax = axes[j]; ax.set_facecolor(BG_COLOR)
        x, y = work[xc], work[yc]
        ax.scatter(x, y, alpha=0.4, s=25, color="C0")
        z = np.polyfit(x, y, 1)
        xl = np.linspace(x.min(), x.max(), 100)
        ax.plot(xl, np.polyval(z, xl), "C3", lw=2.5)
        sig = "***" if pval<.001 else "**" if pval<.01 else "*" if pval<.05 else ""
        ax.set_title(f"{ptitle}\nbeta={beta:.3f}, p={pval:.3f} {sig}", fontsize=10)
        ax.set_xlabel(xc.replace("_r"," (rank)"))
        ax.set_ylabel(yc.replace("_r"," (rank)"))
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig.suptitle(f"{title_str} (n={res["n"]})", fontsize=12, fontweight="bold")
    plt.tight_layout()
    return fig

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RT DESCRIPTIVES
# ══════════════════════════════════════════════════════════════════════

import seaborn as sns
from scipy.stats import spearmanr, pearsonr, rankdata, mannwhitneyu, kruskal

BG_COLOR = "#FAFBFC"
ANX_COLORS = {"Low": "#3498DB", "High": "#E74C3C"}
MODEL_COLORS = {"MF": "#3498DB", "MB": "#2ECC71"}  # blue vs green
MODEL_LABELS = {"MF": "MF-only (M2)",
                "MB": "Model-Based (M3 + M7 + M9)"}
MODEL_SHORT  = {"MF": "MF", "MB": "MB"}
MODEL_GROUPS = ["MF", "MB"]

# ── Motor baseline (SL trials, no planning) ──
# Use the in-memory df (already filtered). No separate filtered_data.csv read.
sl_cols = sorted([c for c in df.columns if c.startswith("SL_T") and ("Step0_RT" in c or "Step1_RT" in c)])
_m = df[["subject_id"] + sl_cols].copy()
_m["subject_id"] = _m["subject_id"].astype(str)
_m["motor_s"] = _m[sl_cols].median(axis=1) / 1000
motor_map = _m.set_index("subject_id")["motor_s"].to_dict()

# ── Prepare all trials ──
rt = trial_df[trial_df["log_rt"].notna()].copy()
rt["subject_id"] = rt["subject_id"].astype(str)
rt["rt_s"] = np.exp(rt["log_rt"]) / 1000
rt["motor_s"] = rt["subject_id"].map(motor_map)
rt["delib_s"] = (rt["rt_s"] - rt["motor_s"].fillna(0)).clip(lower=0.05)
rt["log_delib"] = np.log(rt["delib_s"])

model_map_s = {str(sid): info["model"] for sid, info in winning_params.items()}
rt["model"] = rt["subject_id"].map(model_map_s)
rt["model_label"] = rt["model"].map(MODEL_LABELS)

anx_map = rt.groupby("subject_id")["somatic_anxiety"].first()
med_anx = anx_map.median()
rt["anx_group"] = rt["subject_id"].map(anx_map.apply(lambda a: "High" if a >= med_anx else "Low"))

# Per-subject summaries
subj = rt.groupby("subject_id").agg(
    mean_delib=("delib_s", "mean"), median_delib=("delib_s", "median"),
    mean_log_delib=("log_delib", "mean"), motor_s=("motor_s", "first"),
    mean_rt_s=("rt_s", "mean"), somatic_anxiety=("somatic_anxiety", "first"),
    model=("model", "first"), anx_group=("anx_group", "first"),
    accuracy=("correct", "mean"),
    accuracy_canon=("correct_canon", "mean"),   # PREREG: 0/0.5/1 scoring
).reset_index()

N = rt["subject_id"].nunique()
print(f"RT Descriptives: {len(rt)} trials, {N} subjects")
print(f"  Motor RT: M={subj['motor_s'].mean():.2f}s")
print(f"  Query RT: M={subj['mean_rt_s'].mean():.2f}s")
print(f"  Deliberation: M={subj['mean_delib'].mean():.2f}s")
print("\n  Subjects per winning model:")
for m in MODEL_GROUPS:
    n_m = subj[subj["model"] == m]["subject_id"].nunique()
    print(f"    {MODEL_SHORT[m]:>4s}: n={n_m}")

# ══════════════════════════════════════════════════════════════════════
# PLOT 1: Motor RT distribution
# ══════════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), facecolor=BG_COLOR)

ax = ax1; ax.set_facecolor(BG_COLOR)
ax.hist(subj["motor_s"] * 1000, bins=25, color="C7", alpha=0.7, edgecolor="white")
ax.set_xlabel("Motor RT (ms)")
ax.set_ylabel("N subjects")
ax.set_title("Motor Baseline (SL trials)\nNo planning — pure motor speed", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

ax = ax2; ax.set_facecolor(BG_COLOR)
ax.hist(subj["mean_delib"] * 1000, bins=25, color="C0", alpha=0.7, edgecolor="white")
ax.set_xlabel("Mean Deliberation RT (ms)")
ax.set_title("Query Deliberation (RT − motor)\nPlanning + evaluation time", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(f"fig_rt_distributions_basic_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 2: Deliberation RT distribution by MF / MB
# ══════════════════════════════════════════════════════════════════════

m3 = subj[subj["model"].isin(MODEL_GROUPS)].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG_COLOR)

# Hist
ax = ax1; ax.set_facecolor(BG_COLOR)
for mname in MODEL_GROUPS:
    vals = m3[m3["model"] == mname]["mean_log_delib"]
    ax.hist(vals, bins=15, alpha=0.45, color=MODEL_COLORS[mname],
            label=MODEL_LABELS[mname], edgecolor="white")
ax.set_xlabel("Mean log(Deliberation RT)", fontsize=11)
ax.set_ylabel("N subjects")
ax.set_title("RT Distribution: MF vs MB", fontweight="bold")
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Box + strip
ax = ax2; ax.set_facecolor(BG_COLOR)
m3["model_short"] = m3["model"].map(MODEL_SHORT)
order = [MODEL_SHORT[m] for m in MODEL_GROUPS]
palette = {MODEL_SHORT[m]: MODEL_COLORS[m] for m in MODEL_GROUPS}
sns.boxplot(data=m3, x="model_short", y="mean_log_delib", order=order, palette=palette,
            ax=ax, width=0.5, fliersize=0)
sns.stripplot(data=m3, x="model_short", y="mean_log_delib", order=order, color="black",
              alpha=0.3, size=4, ax=ax, jitter=0.15)
ax.set_xlabel("Model Group")
ax.set_ylabel("Mean log(Deliberation RT)")
ax.set_title("MF vs MB: Deliberation Time", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Kruskal-Wallis across 3 groups
groups_vals = [m3[m3["model"]==m]["mean_log_delib"].values for m in MODEL_GROUPS]
H, p_H = kruskal(*groups_vals)
ax.text(0.5, 0.96, f"Kruskal-Wallis H={H:.2f}, p={p_H:.4f}",
        transform=ax.transAxes, ha="center", va="top", fontsize=10)

# Pairwise Mann-Whitney
print("\nRT by model (pairwise Mann-Whitney on mean_log_delib):")
for a_i in range(len(MODEL_GROUPS)):
    for b_i in range(a_i+1, len(MODEL_GROUPS)):
        a, b = MODEL_GROUPS[a_i], MODEL_GROUPS[b_i]
        u, pu = mannwhitneyu(m3[m3["model"]==a]["mean_log_delib"],
                             m3[m3["model"]==b]["mean_log_delib"])
        print(f"  {MODEL_SHORT[a]} vs {MODEL_SHORT[b]}: U={u:.0f}, p={pu:.4f}")

plt.tight_layout()
plt.savefig(f"fig_rt_by_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 2b (NEW): Anxiety distribution by model group
# ══════════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG_COLOR)

# KDE / hist
ax = ax1; ax.set_facecolor(BG_COLOR)
for mname in MODEL_GROUPS:
    vals = m3[m3["model"] == mname]["somatic_anxiety"].dropna()
    ax.hist(vals, bins=15, alpha=0.45, color=MODEL_COLORS[mname],
            label=f"{MODEL_SHORT[mname]} (n={len(vals)})", edgecolor="white")
ax.set_xlabel("Somatic Anxiety", fontsize=11)
ax.set_ylabel("N subjects")
ax.set_title("Somatic Anxiety Distribution by Model Group", fontweight="bold")
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Box + strip with Kruskal-Wallis
ax = ax2; ax.set_facecolor(BG_COLOR)
sns.boxplot(data=m3.dropna(subset=["somatic_anxiety"]), x="model_short", y="somatic_anxiety",
            order=order, palette=palette, ax=ax, width=0.5, fliersize=0)
sns.stripplot(data=m3.dropna(subset=["somatic_anxiety"]), x="model_short", y="somatic_anxiety",
              order=order, color="black", alpha=0.3, size=4, ax=ax, jitter=0.15)
ax.set_xlabel("Model Group")
ax.set_ylabel("Somatic Anxiety")
ax.set_title("Somatic Anxiety by Model Group", fontweight="bold")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

anx_groups = [m3[m3["model"]==m]["somatic_anxiety"].dropna().values for m in MODEL_GROUPS]
H_a, p_Ha = kruskal(*anx_groups)
ax.text(0.5, 0.96, f"Kruskal-Wallis H={H_a:.2f}, p={p_Ha:.4f}",
        transform=ax.transAxes, ha="center", va="top", fontsize=10)

print("\nSomatic Anxiety by model group:")
for m in MODEL_GROUPS:
    v = m3[m3["model"]==m]["somatic_anxiety"].dropna()
    print(f"  {MODEL_SHORT[m]:>4s} (n={len(v):3d}): M={v.mean():.3f}, SD={v.std():.3f}, "
          f"median={v.median():.3f}")
print(f"  Kruskal-Wallis across 3 groups: H={H_a:.3f}, p={p_Ha:.4f}")
print("  Pairwise Mann-Whitney:")
for a_i in range(len(MODEL_GROUPS)):
    for b_i in range(a_i+1, len(MODEL_GROUPS)):
        a, b = MODEL_GROUPS[a_i], MODEL_GROUPS[b_i]
        va = m3[m3["model"]==a]["somatic_anxiety"].dropna()
        vb = m3[m3["model"]==b]["somatic_anxiety"].dropna()
        u, pu = mannwhitneyu(va, vb)
        print(f"    {MODEL_SHORT[a]} vs {MODEL_SHORT[b]}: U={u:.0f}, p={pu:.4f}")

plt.tight_layout()
plt.savefig(f"fig_anxiety_by_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 3: RT by Anxiety within each model group
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG_COLOR)

for i, mname in enumerate(MODEL_GROUPS):
    ax = axes[i]; ax.set_facecolor(BG_COLOR)
    ms = m3[m3["model"] == mname]
    for anx in ["Low", "High"]:
        vals = ms[ms["anx_group"] == anx]["mean_log_delib"]
        ax.hist(vals, bins=12, alpha=0.5, color=ANX_COLORS[anx],
                label=f"{anx} Somatic Anxiety", edgecolor="white")
    ax.set_xlabel("Mean log(Deliberation RT)")
    if i == 0: ax.set_ylabel("N subjects")
    n_s = len(ms)
    v = ms.dropna(subset=["somatic_anxiety"])
    if len(v) >= 10:
        rho, p = spearmanr(v["somatic_anxiety"], v["mean_log_delib"])
        s = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
        ax.set_title(f"{MODEL_LABELS[mname]} (n={n_s})\nAnxiety~RT: rho={rho:.3f}, p={p:.3f} {s}",
                     fontweight="bold", fontsize=10)
    else:
        ax.set_title(f"{MODEL_LABELS[mname]} (n={n_s})", fontweight="bold", fontsize=10)
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("Deliberation RT by Somatic Anxiety × Model Group", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_rt_by_anxiety_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

# ══════════════════════════════════════════════════════════════════════
# PLOT 4: log(Delib RT) by |EV diff| × Anxiety, for each model group
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG_COLOR)

rt_m = rt[rt["model"].isin(MODEL_GROUPS)].copy()

for i, mname in enumerate(MODEL_GROUPS):
    ax = axes[i]; ax.set_facecolor(BG_COLOR)
    sub = rt_m[rt_m["model"] == mname]
    for anx in ["Low", "High"]:
        anx_sub = sub[sub["anx_group"] == anx]
        if len(anx_sub) == 0: continue
        means = anx_sub.groupby("abs_ev_diff").agg(
            y_mean=("log_delib", "mean"), y_sem=("log_delib", "sem")).reset_index()
        means = means.sort_values("abs_ev_diff")
        ax.errorbar(means["abs_ev_diff"], means["y_mean"], yerr=1.96 * means["y_sem"],
                    color=ANX_COLORS[anx], marker="o", lw=2.5, capsize=4, ms=8,
                    label=f"{anx} Somatic Anxiety", markeredgecolor="white", markeredgewidth=1)
    ax.set_xlabel("|EV − Offer|", fontsize=11)
    if i == 0: ax.set_ylabel("log(Deliberation RT)", fontsize=11)
    ax.set_title(f"{MODEL_LABELS[mname]}", fontweight="bold", fontsize=11)
    ax.legend(fontsize=10)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("Deliberation RT by Difficulty × Somatic Anxiety: MF vs MB\n(RT = query RT − motor baseline)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_rt_by_evdiff_anxiety_model_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()

print("\n" + "=" * 70)
print("ANXIETY ~ DELIBERATION RT CORRELATIONS")
print("=" * 70)
for mname in ["All"] + MODEL_GROUPS:
    s = subj if mname == "All" else subj[subj["model"] == mname]
    v = s.dropna(subset=["somatic_anxiety"])
    if len(v) < 10: continue
    rho, p = spearmanr(v["somatic_anxiety"], v["mean_log_delib"])
    stars = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
    mlbl = mname if mname == "All" else MODEL_SHORT[mname]
    print(f"  {mlbl:>4s} (n={len(v):3d}): rho={rho:+.3f}, p={p:.4f} {stars}")


In [ ]:
# ── Dependency check ──
# This cell needs `subj` (per-subject summary table built in the RT DESCRIPTIVES cell)
# which in turn needs `winning_params` from the CBM/HBI section.
# If you got "NameError: name 'subj' is not defined", run cells in order:
#   - Section 8 (CBM/HBI):   cells 8A through 8D (CBM fits, BMS, HBI)
#   - Section 8E/8F:         PPC + group collapse (MF vs MB)
#   - RT DESCRIPTIVES cell:  defines `subj`
# Then re-run this cell.
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` is not defined. Run the CBM/HBI cells (8A-8D), then the PPC "
        "and group-collapse cells, then the RT DESCRIPTIVES cell before this one."
    )

# ══════════════════════════════════════════════════════════════════════
# OVERTHINKING (preregistered formula)
# ══════════════════════════════════════════════════════════════════════
# Per-subject means only. Z-score ONCE across the full MF+MB sample
# (between-subject standardization), per prereg.
#
#   OT_i = z(delib_log_i) − z(logit(acc_canon_i))
#
# where logit(p) = log(p/(1-p)) with p clipped to [0.025, 0.975].
# Positive OT = OVERTHINKING (slow AND inaccurate).
# Negative OT = efficient (fast AND accurate).
# (This is the sign convention named in the prereg footnote — the prior
#  pilot's OT had the opposite sign; the current convention's predicted
#  H1/H2 correlations are POSITIVE.)

from scipy.stats import spearmanr, pearsonr, rankdata

BG_COLOR = "#FAFBFC"

# PREREG-FIDELITY: pool z-score across the FULL sample (no CBM filter),
# and carry the model-agnostic MB classifier on ot so H1/H2 inherit it.
ot = subj.copy()
ot = ot.merge(mb_score_df[['subject_id', 'mb_classif_agnostic']],
              on='subject_id', how='left')
ot = ot.dropna(subset=["somatic_anxiety", "mean_log_delib", "accuracy_canon"])

# Pooled z-scoring across ALL subjects in MF + MB
# Prereg formula — z(delib_log) − z(logit(acc_canon)), with the logit
# clipped to [0.025, 0.975] to avoid infinities at the extremes.
def _logit_clipped(p, lo=0.025, hi=0.975):
    p_c = np.clip(p, lo, hi)
    return np.log(p_c / (1.0 - p_c))

ot["z_delib_log"] = (ot["mean_log_delib"] - ot["mean_log_delib"].mean()) / ot["mean_log_delib"].std()
ot["logit_acc"]   = _logit_clipped(ot["accuracy_canon"])
ot["z_logit_acc"] = (ot["logit_acc"] - ot["logit_acc"].mean()) / ot["logit_acc"].std()
ot["ot"]          = ot["z_delib_log"] - ot["z_logit_acc"]    # POSITIVE = overthinking
# Keep legacy names available so downstream plot code that references
# z_rt / z_acc still works (z_rt aliases z_delib_log; z_acc aliases the
# new z_logit_acc — both are now under the prereg's transform).
ot["z_rt"]  = ot["z_delib_log"]
ot["z_acc"] = ot["z_logit_acc"]

print(f"Pooled z-scoring across {len(ot)} subjects (MF + MB)")
print(f"  mean_log_delib:   M={ot['mean_log_delib'].mean():+.3f},  SD={ot['mean_log_delib'].std():.3f}")
print(f"  accuracy:         M={ot['accuracy'].mean():+.3f},   SD={ot['accuracy'].std():.3f}")
print(f"  OT (pooled z):    M={ot['ot'].mean():+.3f},   SD={ot['ot'].std():.3f}")

print("\n" + "="*70)
print("OT ~ Anxiety  (per group, pooled z-scoring)")
print("="*70)

rows = []
for grp in MODEL_GROUPS:
    g = ot[ot["model"] == grp].dropna(subset=["somatic_anxiety","ot","z_rt","z_acc"])
    if len(g) < 2:
        rows.append(dict(grp=grp, n=len(g), rho_ot=np.nan, p_ot=np.nan,
                         r_ot=np.nan, p_ot_p=np.nan,
                         rho_rt=np.nan, p_rt=np.nan, rho_acc=np.nan, p_acc=np.nan))
        print(f"\n  {MODEL_LABELS[grp]}  (n={len(g)}) — too few subjects for correlations, skipping")
        continue
    rho_ot,  p_ot_s  = spearmanr(g["somatic_anxiety"], g["ot"])
    r_ot,    p_ot_p  = pearsonr(g["somatic_anxiety"], g["ot"])
    rho_rt,  p_rt    = spearmanr(g["somatic_anxiety"], g["z_rt"])
    rho_acc, p_acc   = spearmanr(g["somatic_anxiety"], g["z_acc"])
    rows.append(dict(grp=grp, n=len(g), rho_ot=rho_ot, p_ot=p_ot_s,
                     r_ot=r_ot, p_ot_p=p_ot_p,
                     rho_rt=rho_rt, p_rt=p_rt, rho_acc=rho_acc, p_acc=p_acc))

    s_ot  = "***" if p_ot_s<.001  else "**" if p_ot_s<.01  else "*" if p_ot_s<.05  else ""
    s_rt  = "***" if p_rt<.001    else "**" if p_rt<.01    else "*" if p_rt<.05    else ""
    s_acc = "***" if p_acc<.001   else "**" if p_acc<.01   else "*" if p_acc<.05   else ""

    print(f"\n  {MODEL_LABELS[grp]}  (n={len(g)})")
    print(f"    Anxiety ~ OT:              rho={rho_ot:+.3f}, p={p_ot_s:.4f} {s_ot:<3s}  "
          f"|  Pearson r={r_ot:+.3f}, p={p_ot_p:.4f}")
    print(f"    Anxiety ~ z(log_delib):    rho={rho_rt:+.3f},  p={p_rt:.4f} {s_rt}")
    print(f"    Anxiety ~ z(accuracy):     rho={rho_acc:+.3f},  p={p_acc:.4f} {s_acc}")
    if p_rt < .05 and p_acc > .1:
        print(f"    → OVERTHINKING signature: anxiety increases RT but NOT accuracy.")
    elif p_rt < .05 and p_acc < .05 and rho_acc > 0:
        print(f"    → Productive deliberation: anxiety increases both RT and accuracy.")
    elif p_rt > .1:
        print(f"    → No anxiety → RT effect in this group.")

# Also a between-groups test: does OT differ between MF and MB?
from scipy.stats import mannwhitneyu
g_mf = ot[ot["model"] == "MF"]["ot"]
g_mb = ot[ot["model"] == "MB"]["ot"]
u, p_u = mannwhitneyu(g_mf, g_mb, alternative="two-sided")
print(f"\n  Between-group OT: MF median={g_mf.median():+.3f}, MB median={g_mb.median():+.3f}")
print(f"    Mann-Whitney U={u:.0f}, p={p_u:.4f}")

# ══════════════════════════════════════════════════════════════════════
# PLOT 1: OT ~ Anxiety (rank-rank), one panel per group + RT/Acc scatter
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG_COLOR)

for j, grp in enumerate(MODEL_GROUPS):
    ax = axes[j]; ax.set_facecolor(BG_COLOR)
    g = ot[ot["model"] == grp]
    xr, yr = rankdata(g["somatic_anxiety"]), rankdata(g["ot"])
    ax.scatter(xr, yr, alpha=0.65, s=50,
               color=MODEL_COLORS[grp], edgecolors="white", linewidths=0.6)
    z = np.polyfit(xr, yr, 1)
    xs = np.linspace(xr.min(), xr.max(), 100)
    ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
    rho, p = spearmanr(g["somatic_anxiety"], g["ot"])
    star = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
    ax.set_xlabel("Anxiety (rank)", fontsize=11)
    if j == 0:
        ax.set_ylabel("OT = z(acc) − z(log Delib) (rank)", fontsize=11)
    ax.set_title(f"{MODEL_LABELS[grp]}\nn={len(g)},  rho={rho:.3f},  p={p:.3f} {star}",
                 fontweight="bold", fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Third panel: z(RT) vs z(acc), colored by anxiety, shaped by group
ax = axes[2]; ax.set_facecolor(BG_COLOR)
markers = {"MF": "o", "MB": "s"}
for grp in MODEL_GROUPS:
    g = ot[ot["model"] == grp]
    sc = ax.scatter(g["z_rt"], g["z_acc"], c=g["somatic_anxiety"], cmap="RdYlBu_r",
                    alpha=0.8, s=65, marker=markers[grp],
                    edgecolors=MODEL_COLORS[grp], linewidths=1.8,
                    label=f"{MODEL_SHORT[grp]} (n={len(g)})")
plt.colorbar(sc, ax=ax, label="Anxiety", shrink=0.8)
ax.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
ax.set_xlabel("z(log Delib RT)   ← faster | slower →", fontsize=10)
ax.set_ylabel("z(accuracy)   ← less | more →", fontsize=10)
ax.set_title("RT × Accuracy (pooled z)\nupper-left = efficient  |  lower-right = overthinking",
             fontweight="bold", fontsize=10)
ax.legend(fontsize=9, loc="best")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Simple OT = z(accuracy) − z(log Delib RT)   —   pooled z, per-group correlation",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_ot_simple_pooled_z_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


In [ ]:
# ── Dependency check ──
# This cell needs `subj` (per-subject summary table built in the RT DESCRIPTIVES cell)
# which in turn needs `winning_params` from the CBM/HBI section.
# If you got "NameError: name 'subj' is not defined", run cells in order:
#   - Section 8 (CBM/HBI):   cells 8A through 8D (CBM fits, BMS, HBI)
#   - Section 8E/8F:         PPC + group collapse (MF vs MB)
#   - RT DESCRIPTIVES cell:  defines `subj`
# Then re-run this cell.
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` is not defined. Run the CBM/HBI cells (8A-8D), then the PPC "
        "and group-collapse cells, then the RT DESCRIPTIVES cell before this one."
    )

# ══════════════════════════════════════════════════════════════════════
# OVERTHINKING — prereg formula, per-subgroup z-scoring (descriptive)
# ══════════════════════════════════════════════════════════════════════
# For the whole group AND for each subgroup in MODEL_GROUPS:
#   1. Take per-subject mean log(deliberation_RT) and mean correct_canon
#   2. Z-score BOTH between subjects WITHIN that analysis group
#   3. OT = z(delib_log) − z(logit(acc_canon))   [prereg formula]
#      Positive OT = overthinking (slow AND inaccurate)
#      Negative OT = efficient    (fast AND accurate)
#   4. Relate OT to anxiety (Spearman + Pearson)
# Note: this cell uses per-subgroup z-scoring, distinct from the H1/H2
# tests (which use pooled MF+MB z-scoring, per prereg).

from scipy.stats import spearmanr, pearsonr, rankdata

BG_COLOR = "#FAFBFC"

# Pool over the subgroups defined in MODEL_GROUPS (from the descriptives cell)
ot_pool = subj[subj["model"].isin(MODEL_GROUPS)].copy()
print(f"OT = z(acc) − z(RT): N pooled = {len(ot_pool)} subjects across {len(MODEL_GROUPS)} subgroups")

# Define analysis groups: 'All' first, then each subgroup
analysis_groups = [("All", None)] + [(MODEL_SHORT[m], m) for m in MODEL_GROUPS]

def _analysis_color(key):
    """All = slate; subgroups take MODEL_COLORS."""
    if key is None:
        return "#34495E"
    return MODEL_COLORS.get(key, "#555555")

def _analysis_label(short, key):
    if key is None:
        return "All subjects (pooled)"
    return f"{short}: {MODEL_LABELS[key]}"

rows = []
for short, mkey in analysis_groups:
    g = ot_pool.copy() if mkey is None else ot_pool[ot_pool["model"] == mkey].copy()
    if len(g) < 10:
        print(f"\n  {short}: n={len(g)}, too few")
        continue

    # Z-score between subjects WITHIN this analysis group
    # Prereg formula (positive OT = overthinking)
    def _logit_clip(p, lo=0.025, hi=0.975):
        p_c = np.clip(p, lo, hi)
        return np.log(p_c / (1.0 - p_c))
    g["z_delib_log"] = (g["mean_log_delib"] - g["mean_log_delib"].mean()) / g["mean_log_delib"].std()
    g["_logit_acc"]  = _logit_clip(g["accuracy_canon"])
    g["z_logit_acc"] = (g["_logit_acc"] - g["_logit_acc"].mean()) / g["_logit_acc"].std()
    g["ot"]          = g["z_delib_log"] - g["z_logit_acc"]   # POSITIVE = overthinking
    g["z_rt"]  = g["z_delib_log"]    # legacy alias for plotting code below
    g["z_acc"] = g["z_logit_acc"]   # legacy alias for plotting code below

    v = g.dropna(subset=["somatic_anxiety"])

    rho_ot,  p_ot   = spearmanr(v["somatic_anxiety"], v["ot"])
    r_ot,    p_ot_p = pearsonr(v["somatic_anxiety"], v["ot"])
    rho_rt,  p_rt   = spearmanr(v["somatic_anxiety"], v["z_rt"])
    rho_acc, p_acc  = spearmanr(v["somatic_anxiety"], v["z_acc"])

    s_ot  = "***" if p_ot<.001  else "**" if p_ot<.01  else "*" if p_ot<.05  else ""
    s_rt  = "***" if p_rt<.001  else "**" if p_rt<.01  else "*" if p_rt<.05  else ""
    s_acc = "***" if p_acc<.001 else "**" if p_acc<.01 else "*" if p_acc<.05 else ""

    print(f"\n  {_analysis_label(short, mkey)} (n={len(v)})")
    print(f"    Anxiety ~ OT = z(acc) − z(RT):  Spearman rho={rho_ot:+.3f} p={p_ot:.4f} {s_ot}  "
          f"Pearson r={r_ot:+.3f} p={p_ot_p:.4f}")
    print(f"      Component → Anxiety ~ z(RT):  rho={rho_rt:+.3f} p={p_rt:.4f} {s_rt}")
    print(f"      Component → Anxiety ~ z(acc): rho={rho_acc:+.3f} p={p_acc:.4f} {s_acc}")

    g["__short"] = short
    g["__mkey"]  = mkey if mkey is not None else "__all__"
    g["__anxiety_rank"] = rankdata(g["somatic_anxiety"].fillna(g["somatic_anxiety"].median()))
    g["__ot_rank"]      = rankdata(g["ot"])
    rows.append((short, mkey, g))

# ── Plot 1: OT ~ Anxiety (rank-rank), one panel per analysis group ──
if len(rows) > 0:
    n_panels = len(rows)
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=BG_COLOR)
    if n_panels == 1:
        axes = [axes]

    for i, (short, mkey, g) in enumerate(rows):
        ax = axes[i]; ax.set_facecolor(BG_COLOR)
        v = g.dropna(subset=["somatic_anxiety", "ot"])
        xr, yr = v["__anxiety_rank"], v["__ot_rank"]
        ax.scatter(xr, yr, alpha=0.55, s=40, color=_analysis_color(mkey),
                   edgecolors="white", linewidths=0.5)
        z = np.polyfit(xr, yr, 1)
        xs = np.linspace(xr.min(), xr.max(), 100)
        ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
        rho, p = spearmanr(v["somatic_anxiety"], v["ot"])
        s = "***" if p<.001 else "**" if p<.01 else "*" if p<.05 else ""
        ax.set_xlabel("Anxiety (rank)", fontsize=11)
        if i == 0:
            ax.set_ylabel("OT = z(acc) − z(RT) (rank)", fontsize=11)
        ax.set_title(f"{_analysis_label(short, mkey)}\nn={len(v)}, rho={rho:.3f}, p={p:.3f} {s}",
                     fontweight="bold", fontsize=10)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle("OT = z(accuracy) − z(log Delib RT), z-scored within group ~ Anxiety\n"
                 "(positive OT = efficient; negative OT = slow + inaccurate = overthinking)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig_ot_zscore_anxiety_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    plt.show()

    # ── Plot 2: z(RT) vs z(acc), colored by anxiety ──
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=BG_COLOR)
    if n_panels == 1:
        axes = [axes]

    for i, (short, mkey, g) in enumerate(rows):
        ax = axes[i]; ax.set_facecolor(BG_COLOR)
        v = g.dropna(subset=["somatic_anxiety"])
        sc = ax.scatter(v["z_rt"], v["z_acc"], c=v["somatic_anxiety"],
                        cmap="RdYlBu_r", alpha=0.7, s=45,
                        edgecolors="white", linewidths=0.5)
        plt.colorbar(sc, ax=ax, label="Anxiety", shrink=0.8)
        ax.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
        ax.axvline(0, color="gray", ls="--", lw=0.8, alpha=0.5)
        ax.set_xlabel("z(log Delib RT)\n← faster  |  slower →", fontsize=10)
        if i == 0:
            ax.set_ylabel("z(accuracy)\n← less accurate  |  more accurate →", fontsize=10)
        ax.set_title(f"{_analysis_label(short, mkey)}\nupper-left = efficient | lower-right = overthinking",
                     fontweight="bold", fontsize=10)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle("RT vs Accuracy (z-scored within group), colored by anxiety",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"fig_ot_zscore_components_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
    plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H1 (preregistered) — Somatic Anxiety × Overthinking
# ══════════════════════════════════════════════════════════════════════
# Predictor : Survey_ANXIETY_2_Total_Score (10-item Physiological Anxiety
#             Checklist = somatic anxiety), assigned to df['somatic_anxiety']
#             in cell 3 of the V4 notebook (post-rename 2026-05-27).
# Outcome   : OT = z(delib_log) − z(logit(acc_canon)), pooled-z across the
#             full MF+MB sample, computed in the simple-OT cell above and
#             held in `ot`. Positive OT = overthinking (prereg sign).
# Test      : Spearman ρ, one-sided positive p (per prereg, α = .05).
# Partition : mb_classif_agnostic — model-agnostic MB classifier per prereg.
# Panels    : All, mb_score-MB (PRIMARY, H1), mb_score-nonMB,
#             plus CBM MF / CBM MB as exploratory descriptives.
#
# Additive cell: parallels the H2 cell but substitutes the somatic-anxiety
# predictor. Does not modify any earlier analysis.
# ══════════════════════════════════════════════════════════════════════
if "ot" not in dir():
    raise RuntimeError("`ot` not defined. Run the simple OT × Anxiety cell first.")
if "somatic_anxiety" not in df.columns:
    raise RuntimeError("df['somatic_anxiety'] missing — re-run cell 3 (it sets "
                       "Anxiety = Survey_ANXIETY_2_Total_Score, the somatic scale).")
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score classifier cell.")

from scipy.stats import spearmanr, pearsonr, rankdata

# Join somatic anxiety into the OT table
somatic_map = df.set_index("subject_id")["somatic_anxiety"]
ot_h1 = ot.copy()
ot_h1["somatic_anxiety"] = ot_h1["subject_id"].map(somatic_map)
ot_h1 = ot_h1.dropna(subset=["somatic_anxiety", "ot"])

# Model-agnostic partition inherited from `ot` (built in cell 36).

print("=" * 70)
print("H1 — Somatic Anxiety × OT   (prereg: one-sided positive Spearman, α=.05)")
print("=" * 70)
print(f"Sample after dropping NaN somatic/OT: N = {len(ot_h1)}")

analysis_groups_h1 = [
    ("All",            "all",    None,  False),
    ("MB (mb_score)",  "mb_agn", True,  True),    # ← prereg primary
    ("nonMB (mb_score)","mb_agn", False, False),
    ("MF  (CBM)",      "cbm",    "MF",  False),   # exploratory
    ("MB  (CBM)",      "cbm",    "MB",  False),   # exploratory
]
rows_h1 = []
for label, kind, val, is_primary in analysis_groups_h1:
    if kind == "all":
        g = ot_h1
    elif kind == "mb_agn":
        g = ot_h1[ot_h1["mb_classif_agnostic"] == val]
    else:   # cbm
        g = ot_h1[ot_h1["model"] == val]
    if len(g) < 3:
        print(f"\n  {label:17s} (n={len(g)}) — too few for correlation")
        rows_h1.append((label, is_primary, g, np.nan, np.nan, np.nan, np.nan))
        continue
    rho, p_two = spearmanr(g["somatic_anxiety"], g["ot"])
    p_one = (p_two / 2) if rho > 0 else (1 - p_two / 2)
    r, p_r_two = pearsonr(g["somatic_anxiety"], g["ot"])
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    h1_tag = "   ← H1 (preregistered)" if is_primary else ""
    print(f"\n  {label:17s} (n={len(g):3d}){h1_tag}")
    print(f"    Spearman rho = {rho:+.3f},  two-sided p = {p_two:.4f},  one-sided p = {p_one:.4f} {star}")
    print(f"    Pearson  r   = {r:+.3f},   two-sided p = {p_r_two:.4f}")
    rows_h1.append((label, is_primary, g, rho, p_one, r, p_r_two))

# ── Plot ──
_BG_H1 = BG_COLOR if "BG_COLOR" in dir() else "#FAFBFC"
GROUP_COLORS_H1 = {
    "All":              "#34495E",
    "MB (mb_score)":    "#E74C3C",    # red — primary
    "nonMB (mb_score)": "#F1948A",
    "MF  (CBM)":        "#FAD7A0",    # exploratory
    "MB  (CBM)":        "#F5CBA7",    # exploratory
}

n_panels = len(rows_h1)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=_BG_H1)
if n_panels == 1:
    axes = [axes]

for j, (label, is_primary, g, rho, p_one, r, p_r_two) in enumerate(rows_h1):
    ax = axes[j]; ax.set_facecolor(_BG_H1)
    ax.set_xlabel("Somatic Anxiety (rank)", fontsize=11)
    if j == 0:
        ax.set_ylabel("OT = z(delib_log) − z(logit acc) (rank)", fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    h1_tag = "  ← H1" if is_primary else ""
    if len(g) < 3:
        ax.text(0.5, 0.5, f"n = {len(g)}\ntoo few", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
        ax.set_title(f"{label}{h1_tag}\n(n={len(g)})", fontweight="bold", fontsize=11)
        continue
    xr, yr = rankdata(g["somatic_anxiety"]), rankdata(g["ot"])
    ax.scatter(xr, yr, alpha=0.65, s=50,
               color=GROUP_COLORS_H1.get(label, "#555555"),
               edgecolors="white", linewidths=0.6)
    z = np.polyfit(xr, yr, 1)
    xs = np.linspace(xr.min(), xr.max(), 100)
    ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    ax.set_title(f"{label}{h1_tag}\nn={len(g)}, rho={rho:.3f}, one-sided p={p_one:.4f} {star}",
                 fontweight="bold", fontsize=11)

fig.suptitle("H1 — Somatic Anxiety × Overthinking\n"
             "Preregistered: positive Spearman within mb_score-classified MB participants (one-sided, α=.05).\n"
             "CBM-group panels are exploratory.",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_h1_somatic_ot_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=_BG_H1)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H2 (preregistered) — Worry × Overthinking
# ══════════════════════════════════════════════════════════════════════
# Additive cell: does not modify any earlier analysis. Parallels the
# simple OT × Anxiety cell above but substitutes the Worry predictor.
#
# Predictor : Survey_ANXIETY_1_Total_Score (8-item PSWQ trait Worry),
#             already assigned to df["worry"] in cell 3.
# Outcome   : OT = z(delib_log) − z(logit(acc_canon)), pooled-z across
#             MF+MB, computed in the simple-OT cell above and held in `ot`.
#             Positive OT = overthinking (prereg sign convention), so the
#             predicted one-sided positive Spearman ρ tests the H2 claim
#             that higher worry → more overthinking.
# Test      : Spearman rho with a one-sided positive p (per prereg, α=.05).
# Panels    : All, mb_score-MB (PRIMARY, H2), mb_score-nonMB,
#             plus CBM MF / CBM MB as exploratory descriptives.
#             Per prereg, H2 is the mb_score-MB panel.
# ══════════════════════════════════════════════════════════════════════
if "ot" not in dir():
    raise RuntimeError(
        "`ot` not defined. Run the simple OT × Anxiety cell first "
        "(it builds the pooled-z OT table this analysis reuses).")
if "worry" not in df.columns:
    raise RuntimeError(
        "df['worry'] missing — re-run cell 3 (it sets "
        "worry = sum of PSWQ-8 items = Survey_ANXIETY_1_Total_Score).")

from scipy.stats import spearmanr, pearsonr, rankdata

# Join Worry into the OT table
worry_map = df.set_index("subject_id")["worry"]
ot_h2 = ot.copy()
ot_h2["worry"] = ot_h2["subject_id"].map(worry_map)
ot_h2 = ot_h2.dropna(subset=["worry", "ot"])

# Model-agnostic partition inherited from `ot` (built in cell 36).
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score "
                       "classifier cell near the top of the notebook.")

print("=" * 70)
print("H2 — Worry (PSWQ trait) × OT   (prereg: one-sided positive Spearman, α=.05)")
print("=" * 70)
print(f"Sample after dropping NaN Worry/OT: N = {len(ot_h2)}")

# Panel definitions: (label, partition_kind, partition_value, is_h2_primary)
#   partition_kind ∈ {'all', 'mb_agn', 'cbm'}
analysis_groups_h2 = [
    ("All",            "all",    None,  False),
    ("MB (mb_score)",  "mb_agn", True,  True),    # ← prereg primary
    ("nonMB (mb_score)","mb_agn", False, False),
    ("MF  (CBM)",      "cbm",    "MF",  False),   # exploratory
    ("MB  (CBM)",      "cbm",    "MB",  False),   # exploratory
]
rows_h2 = []
for label, kind, val, is_primary in analysis_groups_h2:
    if kind == "all":
        g = ot_h2
    elif kind == "mb_agn":
        g = ot_h2[ot_h2["mb_classif_agnostic"] == val]
    else:   # cbm
        g = ot_h2[ot_h2["model"] == val]
    if len(g) < 3:
        print(f"\n  {label:17s} (n={len(g)}) — too few for correlation")
        rows_h2.append((label, is_primary, g, np.nan, np.nan, np.nan, np.nan))
        continue
    rho, p_two = spearmanr(g["worry"], g["ot"])
    # one-sided positive: halve two-sided when rho>0, else 1 - half
    p_one = (p_two / 2) if rho > 0 else (1 - p_two / 2)
    r, p_r_two = pearsonr(g["worry"], g["ot"])
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    h2_tag = "   ← H2 (preregistered)" if is_primary else ""
    print(f"\n  {label:17s} (n={len(g):3d}){h2_tag}")
    print(f"    Spearman rho = {rho:+.3f},  two-sided p = {p_two:.4f},  one-sided p = {p_one:.4f} {star}")
    print(f"    Pearson  r   = {r:+.3f},   two-sided p = {p_r_two:.4f}")
    rows_h2.append((label, is_primary, g, rho, p_one, r, p_r_two))

# ── Plot: rank-rank scatter, one panel per analysis group ──
_BG_H2 = BG_COLOR if "BG_COLOR" in dir() else "#FAFBFC"
GROUP_COLORS_H2 = {
    "All":              "#34495E",
    "MB (mb_score)":    "#2ECC71",    # green — primary
    "nonMB (mb_score)": "#3498DB",
    "MF  (CBM)":        "#85C1E9",    # light blue — exploratory
    "MB  (CBM)":        "#A9DFBF",    # light green — exploratory
}

n_panels = len(rows_h2)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5), facecolor=_BG_H2)
if n_panels == 1:
    axes = [axes]

for j, (label, is_primary, g, rho, p_one, r, p_r_two) in enumerate(rows_h2):
    ax = axes[j]; ax.set_facecolor(_BG_H2)
    ax.set_xlabel("Worry / PSWQ (rank)", fontsize=11)
    if j == 0:
        ax.set_ylabel("OT = z(delib_log) − z(logit acc) (rank)", fontsize=11)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    h2_tag = "  ← H2" if is_primary else ""
    if len(g) < 3:
        ax.text(0.5, 0.5, f"n = {len(g)}\ntoo few", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
        ax.set_title(f"{label}{h2_tag}\n(n={len(g)})", fontweight="bold", fontsize=11)
        continue
    xr, yr = rankdata(g["worry"]), rankdata(g["ot"])
    ax.scatter(xr, yr, alpha=0.65, s=50,
               color=GROUP_COLORS_H2.get(label, "#555555"),
               edgecolors="white", linewidths=0.6)
    z = np.polyfit(xr, yr, 1)
    xs = np.linspace(xr.min(), xr.max(), 100)
    ax.plot(xs, np.polyval(z, xs), "C3", lw=2.5)
    star = "***" if p_one < .001 else "**" if p_one < .01 else "*" if p_one < .05 else ""
    ax.set_title(f"{label}{h2_tag}\nn={len(g)}, rho={rho:.3f}, one-sided p={p_one:.4f} {star}",
                 fontweight="bold", fontsize=11)

fig.suptitle("H2 — Worry (PSWQ trait) × Overthinking\n"
             "Preregistered: positive Spearman within mb_score-classified MB participants (one-sided, α=.05).\n"
             "CBM-group panels are exploratory.",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_h2_worry_ot_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=_BG_H2)
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H3 (preregistered) — log-deliberation RT, MB vs non-MB
# ══════════════════════════════════════════════════════════════════════
# Prereg H3:
#   Per-subject mean log-transformed deliberation RT (delib_log_i) is larger
#   in MODEL-BASED-classified subjects than in NON-MODEL-BASED subjects.
#
# Test (per prereg Analysis Plan):
#   One-sided Mann-Whitney U on delib_log_i, alternative='greater' (MB > non-MB).
#   Rank-biserial effect size:  r_rb = 1 − 2·U / (n_MB · n_nonMB).
#   Linear-regression sensitivity (Welch-corrected SE) reported alongside.
#
# Partition (per prereg Variables): MODEL-AGNOSTIC mb_score classifier,
# computed in the mb_score classifier cell near the top of the notebook
# (column `mb_classif_agnostic` in `mb_score_df`). The CBM/HBI partition
# (`subj.model`) is NOT used here — that one feeds the convergent-validity
# κ check separately.
#
# Additive cell: does not modify any earlier analysis.
# ══════════════════════════════════════════════════════════════════════
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` not defined. Run the CBM/HBI cells, the PPC + group-collapse "
        "cells, then the RT DESCRIPTIVES cell before this one.")
if 'mb_score_df' not in dir():
    raise RuntimeError(
        "`mb_score_df` not defined. Run the model-agnostic mb_score "
        "classifier cell (near the top of the notebook, right after the "
        "trial-level dataframe build) before this one.")

from scipy.stats import mannwhitneyu
import statsmodels.api as sm

# Build H3 frame: subject × {mean_log_delib, mb_classif_agnostic}
h3 = (subj[['subject_id', 'mean_log_delib']]
      .merge(mb_score_df[['subject_id', 'mb_classif_agnostic',
                          'mb_score_F', 'mb_score_T']],
             on='subject_id', how='inner')
      .dropna(subset=['mean_log_delib', 'mb_classif_agnostic']))

mb_grp    = h3.loc[h3['mb_classif_agnostic'],  'mean_log_delib'].values
nonmb_grp = h3.loc[~h3['mb_classif_agnostic'], 'mean_log_delib'].values
n_mb, n_nonmb = len(mb_grp), len(nonmb_grp)

print("=" * 70)
print("H3 (preregistered) — log-deliberation RT, MB vs non-MB (model-agnostic)")
print("=" * 70)
print(f"  N total      : {n_mb + n_nonmb}")
print(f"  N MB         : {n_mb}")
print(f"  N non-MB     : {n_nonmb}")
if n_mb >= 1 and n_nonmb >= 1:
    print(f"  mean log_delib  MB     : {mb_grp.mean():+.3f}  (SD={mb_grp.std(ddof=1):.3f})")
    print(f"  mean log_delib  non-MB : {nonmb_grp.mean():+.3f}  (SD={nonmb_grp.std(ddof=1):.3f})")

# ── One-sided Mann-Whitney (MB greater than non-MB) ──
if n_mb >= 2 and n_nonmb >= 2:
    u_stat, p_one = mannwhitneyu(mb_grp, nonmb_grp, alternative='greater')
    _, p_two      = mannwhitneyu(mb_grp, nonmb_grp, alternative='two-sided')
    r_rb = 1.0 - (2.0 * u_stat) / (n_mb * n_nonmb)
    star = '***' if p_one < .001 else '**' if p_one < .01 else '*' if p_one < .05 else ''
    print()
    print(f"  Mann-Whitney U (MB > non-MB) : U = {u_stat:.1f}")
    print(f"  one-sided p                  : {p_one:.4f} {star}")
    print(f"  two-sided p (for reference)  : {p_two:.4f}")
    print(f"  rank-biserial r_rb           : {r_rb:+.3f}")
else:
    print()
    print("  Mann-Whitney : insufficient subjects in one group (need ≥ 2 each).")

# ── Welch-corrected OLS sensitivity ──
if n_mb >= 2 and n_nonmb >= 2:
    h3_ols = h3.copy()
    h3_ols['mb'] = h3_ols['mb_classif_agnostic'].astype(int)
    X = sm.add_constant(h3_ols['mb'])
    model_ols = sm.OLS(h3_ols['mean_log_delib'], X).fit(cov_type='HC3')
    print()
    print("  OLS sensitivity (Welch-style HC3 SE):  mean_log_delib ~ mb_group")
    print("  " + str(model_ols.summary2().tables[1]).replace('\n', '\n  '))

# ── Quick visual ──
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.5))
ax.set_facecolor('#FAFBFC')
positions = [0, 1]
bp = ax.boxplot([nonmb_grp, mb_grp], positions=positions, widths=0.55,
                patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='white',
                               markeredgecolor='black', markersize=7))
colors = ['#3498DB', '#2ECC71']
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.55)
# Scatter
rng = np.random.default_rng(0)
for i, g in enumerate([nonmb_grp, mb_grp]):
    x_jit = positions[i] + rng.uniform(-0.13, 0.13, size=len(g))
    ax.scatter(x_jit, g, color=colors[i], edgecolors='white',
               s=42, alpha=0.85, linewidths=0.7)
ax.set_xticks(positions)
ax.set_xticklabels([f'non-MB\n(n={n_nonmb})', f'MB\n(n={n_mb})'])
ax.set_ylabel('mean log(deliberation RT)')
title = 'H3 — MB vs non-MB log-deliberation RT (model-agnostic partition)'
if n_mb >= 2 and n_nonmb >= 2:
    title += f'\nU={u_stat:.0f}, one-sided p={p_one:.4f}, r_rb={r_rb:+.3f}'
ax.set_title(title, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f"fig_h3_log_delib_mb_vs_nonmb_{OUT_PREFIX}.png", dpi=200,
            bbox_inches='tight', facecolor='#FAFBFC')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Convergent validity (preregistered) — Cohen's κ between classifiers
# ROBUST VERSION (2026-07-02): computes the HBI allocation directly from
# the responsibility matrix in cbm_output_<OUT_PREFIX>/hbi_choice.pkl
# instead of relying on in-memory `subj`/`winning_params` state, and uses
# the prereg-correct FULL 11-model collapse (nothing dropped, N = 277).
# The old in-memory path silently dropped subjects whose winner was
# outside {M2,M3,M7,M9} → stale N=118, κ=0.966. Verified externally:
# κ = +0.764, agreement 246/277 (88.8%) — see
# verify_convergence_agnostic_vs_hbi.py / LOGBOOK 2026-07-02.
#
# Prereg flag: if κ < 0.5, the two classifiers no longer measure the
# same construct → reconsider before interpreting H1/H2/H3.
# V3 pilot reference: 85.6% agreement, κ = 0.71 (N = 139).
# ══════════════════════════════════════════════════════════════════════
if 'mb_score_df' not in dir():
    raise RuntimeError("`mb_score_df` not defined — run the mb_score classifier cell.")
if 'trial_df' not in dir():
    raise RuntimeError("`trial_df` not defined — run the trial-level data cell.")

import os, sys, types, pickle
from scipy.stats import spearmanr

# ── 1. Load HBI responsibilities (lenient unpickler: no cbm import needed) ──
_cbm_dir = f"cbm_output_{OUT_PREFIX}" if 'OUT_PREFIX' in dir() else "cbm_output_v4_1_all"
_hbi_path = os.path.join(_cbm_dir, "hbi_choice.pkl")

class _StubObj:
    def __init__(self, *a, **kw): pass
    def __setstate__(self, st):
        self.__dict__.update(st if isinstance(st, dict) else {"_state": st})

class _LenientUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except Exception:
            mod = sys.modules.setdefault(module, types.ModuleType(module))
            if not hasattr(mod, name):
                setattr(mod, name, type(name, (_StubObj,), {}))
            return getattr(mod, name)

with open(_hbi_path, "rb") as _f:
    _hbi = _LenientUnpickler(_f).load()

_resp   = _hbi.output.responsibility          # (N, K)
_mnames = [m if isinstance(m, str) else getattr(m, "__name__", type(m).__name__)
           for m in _hbi.input.models]
print(f"HBI responsibilities loaded from {_hbi_path}: {_resp.shape}, "
      f"models = {_mnames}")

# ── 2. Subject order = CBM fit order (mirror of the data_list cell) ──
if 'FIT_ALL_TRIALS' in dir() and not FIT_ALL_TRIALS:
    _fit = trial_df[trial_df["is_anti_mf"] == 1].dropna(subset=["choice"])
else:
    _fit = trial_df.dropna(subset=["choice"])
_sids = _fit["subject_id"].unique().tolist()
assert len(_sids) == _resp.shape[0], (
    f"alignment failure: {len(_sids)} subjects in fit order vs "
    f"{_resp.shape[0]} rows in HBI responsibility matrix")

# ── 3. Full prereg-correct MF/MB collapse (MF iff no EV term; hybrids→MB) ──
_GROUP_MAP_FN = {
    "m1_mf_only": "MF", "m2_mf_alpha": "MF",
    "m3_mb_only": "MB", "m4_mb_mf": "MB", "m5_mb_sep_stim": "MB",
    "m7_mb_sep_reval": "MB", "m9_sep_w_stim": "MB", "m10_sep_w_reval": "MB",
    "m11_sep_beta_w_reval": "MB", "m12_mb_mf_sep_stim": "MB",
    "m16_sep_beta_w_stim": "MB",
}
def _to_group(name):
    if name in _GROUP_MAP_FN: return _GROUP_MAP_FN[name]
    low = name.lower()
    if "mb" in low or "w_" in low or "det" in low: return "MB"
    if "mf" in low: return "MF"
    raise ValueError(f"unmapped model name: {name}")

_winner   = _resp.argmax(axis=1)
_max_resp = _resp[np.arange(len(_winner)), _winner]
_win_nm   = [_mnames[k] for k in _winner]
_grp      = np.array([_to_group(n) for n in _win_nm])
_mb_cols  = [i for i, n in enumerate(_mnames) if _to_group(n) == "MB"]
_fam_mb   = _resp[:, _mb_cols].sum(axis=1)     # P(MB family) per subject

hbi_alloc = pd.DataFrame({
    "subject_id":  _sids,
    "hbi_model":   _win_nm,
    "hbi_group":   _grp,
    "hbi_max_resp": _max_resp,
    "hbi_P_MB_family": _fam_mb,
})
print(f"HBI collapse: MB = {(_grp=='MB').sum()}, MF = {(_grp=='MF').sum()} "
      f"(N = {len(_grp)}); mean max responsibility = {_max_resp.mean():.3f}")

# ── 4. κ against the model-agnostic classifier (all subjects, none dropped) ──
kdf = (mb_score_df[["subject_id", "mb_score_F", "mb_score_T",
                    "mb_classif_agnostic"]]
       .merge(hbi_alloc, on="subject_id", how="inner"))
kdf["agn_mb"] = kdf["mb_classif_agnostic"].astype(int)
kdf["cbm_mb"] = (kdf["hbi_group"] == "MB").astype(int)

n_joint = len(kdf)
n_agree = int((kdf["agn_mb"] == kdf["cbm_mb"]).sum())
obs     = n_agree / n_joint
_pa, _pc = kdf["agn_mb"].mean(), kdf["cbm_mb"].mean()
exp     = _pa*_pc + (1-_pa)*(1-_pc)
kappa   = (obs - exp) / (1 - exp)
kappa_se = np.sqrt(obs*(1-obs)/n_joint) / (1-exp)
ci_lo, ci_hi = kappa - 1.96*kappa_se, min(kappa + 1.96*kappa_se, 1.0)

print("=" * 70)
print("Convergent validity — model-agnostic vs HBI/CBM (full collapse)")
print("=" * 70)
print(f"  N with both classifications : {n_joint}")
print(f"  Agreement                   : {n_agree}/{n_joint} ({100*obs:.1f}%)")
print(f"  Cohen's κ                   : {kappa:+.3f}  (95% CI {ci_lo:.3f} to {ci_hi:.3f})")
print(f"  Prior-pilot reference (V3)  : 85.6% agreement, κ = 0.71  (N = 139)")
if kappa < 0.5:
    print("  ⚠ κ < 0.5 — per prereg, reconsider before interpreting H1/H2/H3.")
elif kappa < 0.71:
    print("  • κ ≥ 0.5 (prereg OK); slightly below/near V3 pilot 0.71." if kappa < 0.71 else "")
else:
    print("  ✓ κ ≥ V3 pilot — classifiers agree at least as well as in V3.")

# 2×2 confusion
cm = np.zeros((2, 2), dtype=int)
for _c, _a in zip(kdf["cbm_mb"], kdf["agn_mb"]):
    cm[_c, _a] += 1
print()
print("  Confusion matrix  (rows: HBI/CBM, cols: model-agnostic)")
print("                       agn=nonMB   agn=MB")
print(f"    HBI = MF (pure)      {cm[0,0]:4d}       {cm[0,1]:4d}")
print(f"    HBI = MB (+hybrid)   {cm[1,0]:4d}       {cm[1,1]:4d}")

# Continuous convergence: graded mb_score vs graded HBI family probability
kdf["mb_score_max"] = kdf[["mb_score_F", "mb_score_T"]].max(axis=1)
_rho, _p = spearmanr(kdf["mb_score_max"], kdf["hbi_P_MB_family"])
print(f"\n  Continuous check: Spearman(mb_score_max, P(MB family)) "
      f"ρ = {_rho:+.3f}, p = {_p:.3g}")

# Disagreement anatomy (hybrid caveat: MB-by-identity with possibly w≈0)
_dis = kdf[kdf["agn_mb"] != kdf["cbm_mb"]]
print(f"\n  Disagreements: {len(_dis)} "
      f"(HBI-MB/agn-nonMB: {int(((_dis['cbm_mb']==1)).sum())}, "
      f"HBI-MF/agn-MB: {int(((_dis['cbm_mb']==0)).sum())})")
if len(_dis):
    print(_dis.groupby("hbi_model")
              .agg(n=("agn_mb", "size"),
                   mean_max_resp=("hbi_max_resp", "mean"),
                   mean_P_MB_fam=("hbi_P_MB_family", "mean"))
              .round(3).to_string())

# ── 5. Plot + save ──
fig, ax = plt.subplots(1, 1, figsize=(5.5, 4.5))
ax.set_facecolor('#FAFBFC')
im = ax.imshow(cm, cmap='YlGnBu', aspect='auto')
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-MB', 'MB'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['non-MB', 'MB'])
ax.set_xlabel('Model-agnostic classifier')
ax.set_ylabel('CBM (HBI) classifier')
for _i in range(2):
    for _j in range(2):
        ax.text(_j, _i, str(cm[_i, _j]), ha='center', va='center',
                fontsize=16, fontweight='bold',
                color='white' if cm[_i, _j] > cm.max()/2 else 'black')
ax.set_title(f"Convergent validity — Cohen's κ = {kappa:+.3f}\n"
             f"({100*obs:.1f}% agreement, N = {n_joint})",
             fontweight='bold')
plt.tight_layout()
plt.savefig(f"fig_kappa_convergent_validity_{OUT_PREFIX if 'OUT_PREFIX' in dir() else 'v4_1_all'}_CORRECTED.png",
            dpi=180, bbox_inches='tight')
plt.show()

kdf.to_csv("convergence_agnostic_vs_hbi_"
           f"{OUT_PREFIX if 'OUT_PREFIX' in dir() else 'v4_1_all'}.csv", index=False)
print("\nSaved corrected κ figure + convergence CSV.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Sensitivity: hybrid winners reclassified by FITTED w (resolves the
# 2026-06-10 open item "hybrid-by-identity vs hybrid-by-w")
# ══════════════════════════════════════════════════════════════════════
# The identity rule labels every hybrid winner MB even when fitted w≈0
# (i.e. the subject puts ~all decision weight on the cached MF value).
# Here each hybrid winner is relabeled MB iff any of their fitted mixing
# weights exceeds 0.5 (OR-rule, mirroring the agnostic classifier).
# w extraction per likelihood (cell 8A): m4 w=expit(p1); m9 wf/wt=expit(p1,p2);
# m10 w1/w2=expit(p1,p2); m11 w=p2 RAW (no sigmoid in likelihood);
# m16 wf/wt=expit(p2,p3); m12 malformed (w=beta_f bug) → kept MB by identity.
# Requires the robust κ cell above (uses _hbi, _mnames, _resp, _winner, kdf).
# Verified 2026-07-02: κ 0.764 → 0.933; splits converge to 160/117 vs 159/118.
# ══════════════════════════════════════════════════════════════════════
_params = _hbi.output.parameters
_midx = {n: i for i, n in enumerate(_mnames)}

def _w_max(sub_i, model):
    p = _params[_midx[model]][sub_i]
    if model == "m4_mb_mf":             return expit(p[1])
    if model == "m9_sep_w_stim":        return max(expit(p[1]), expit(p[2]))
    if model == "m10_sep_w_reval":      return max(expit(p[1]), expit(p[2]))
    if model == "m11_sep_beta_w_reval": return p[2]      # raw w in likelihood
    if model == "m12_mb_mf_sep_stim":   return np.nan    # malformed w (bug)
    if model == "m16_sep_beta_w_stim":  return max(expit(p[2]), expit(p[3]))
    return None                                          # not a hybrid

_w_df = pd.DataFrame({
    "subject_id": _sids,
    "w_fitted": [_w_max(i, m) for i, m in enumerate(_win_nm)],
})
kdf_w = kdf.merge(_w_df, on="subject_id", how="left")
_is_hyb = kdf_w["hbi_model"].isin(
    ["m4_mb_mf", "m9_sep_w_stim", "m10_sep_w_reval",
     "m11_sep_beta_w_reval", "m12_mb_mf_sep_stim", "m16_sep_beta_w_stim"])

kdf_w["hbi_group_w"] = np.where(
    ~_is_hyb, kdf_w["hbi_group"],
    np.where(kdf_w["w_fitted"].isna(), "MB",           # m12 kept by identity
             np.where(kdf_w["w_fitted"] > 0.5, "MB", "MF")))

_h = kdf_w[_is_hyb & kdf_w["w_fitted"].notna()]
print("Fitted w among hybrid winners (max across w-params):")
for _lab, _sub in [("agnostic-MB", _h[_h["agn_mb"] == 1]),
                   ("agnostic-nonMB", _h[_h["agn_mb"] == 0])]:
    print(f"  {_lab:15s} n={len(_sub):3d}  mean w={_sub['w_fitted'].mean():.3f} "
          f"median={_sub['w_fitted'].median():.3f}  "
          f"w<=0.5: {(_sub['w_fitted']<=.5).sum()}  w>0.5: {(_sub['w_fitted']>.5).sum()}")
_rho_w, _p_w = spearmanr(_h["w_fitted"], _h["mb_score_max"])
print(f"  Spearman(w, mb_score_max) among hybrid winners: "
      f"ρ = {_rho_w:+.3f}, p = {_p_w:.3g}")

_cw = (kdf_w["hbi_group_w"] == "MB").astype(int)
_aw = kdf_w["agn_mb"]
_obs = (_aw == _cw).mean()
_exp = _aw.mean()*_cw.mean() + (1-_aw.mean())*(1-_cw.mean())
_kap_w = (_obs - _exp) / (1 - _exp)
_se_w = np.sqrt(_obs*(1-_obs)/len(kdf_w)) / (1-_exp)
print(f"\nHybrid-by-w split: MB = {(kdf_w['hbi_group_w']=='MB').sum()}, "
      f"MF = {(kdf_w['hbi_group_w']=='MF').sum()}   "
      f"(identity rule: {(kdf_w['hbi_group']=='MB').sum()}/"
      f"{(kdf_w['hbi_group']=='MF').sum()}; agnostic: "
      f"{int(_aw.sum())}/{int((1-_aw).sum())})")
print(f"κ (identity rule)   : {kappa:+.3f}  ({100*obs:.1f}% agreement)")
print(f"κ (fitted-w rule)   : {_kap_w:+.3f}  (95% CI {_kap_w-1.96*_se_w:.3f} to "
      f"{min(_kap_w+1.96*_se_w,1):.3f}; {100*_obs:.1f}% agreement; N = {len(kdf_w)})")
_rem = kdf_w[(_cw != _aw)]
print(f"Remaining disagreements: {len(_rem)}")
if len(_rem):
    print(_rem.groupby("hbi_model").size().to_string())

kdf_w.to_csv(f"hybrid_w_sensitivity_"
             f"{OUT_PREFIX if 'OUT_PREFIX' in dir() else 'v4_1_all'}.csv", index=False)
print("\nSaved hybrid-w sensitivity CSV.")

In [ ]:
# ── Dependency check ──
# This cell needs `subj` (per-subject summary table built in the RT DESCRIPTIVES cell)
# which in turn needs `winning_params` from the CBM/HBI section.
# If you got "NameError: name 'subj' is not defined", run cells in order:
#   - Section 8 (CBM/HBI):   cells 8A through 8D (CBM fits, BMS, HBI)
#   - Section 8E/8F:         PPC + group collapse (MF vs MB)
#   - RT DESCRIPTIVES cell:  defines `subj`
# Then re-run this cell.
if 'subj' not in dir():
    raise RuntimeError(
        "`subj` is not defined. Run the CBM/HBI cells (8A-8D), then the PPC "
        "and group-collapse cells, then the RT DESCRIPTIVES cell before this one."
    )

# ══════════════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS: Robustness of OT ~ Anxiety to HBI assignment uncertainty
# ══════════════════════════════════════════════════════════════════════
# HBI responsibility assignments can shift when the candidate model set changes,
# and borderline subjects (low max responsibility) are especially unstable.
# This cell tests whether the OT~anxiety finding (especially for MB) survives
# reassignment of uncertain subjects. Three scenarios:
#   A. Decisive-only   — drop subjects with max(resp) < TAU
#   B. Second-best flip — reassign borderline subjects to 2nd-best model
#   C. Monte Carlo     — sample assignments from responsibility distribution

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

TAU    = 0.80   # confidence threshold for "decisive"
N_BOOT = 1000   # Monte Carlo draws
SEED   = 42

GROUPS_ALL = MODEL_GROUPS + ["__pool__"]

def _label(mk):
    return "pooled (MF+MB)" if mk == "__pool__" else mk

def run_ot_from_map(model_map, subj_base=subj):
    """Given {subject_id_str: model_name}, rerun OT = z(acc) - z(RT) vs anxiety per group."""
    s = subj_base.copy()
    s["model"] = s["subject_id"].astype(str).map(model_map)
    out = {}
    for mk in GROUPS_ALL:
        g = (s[s["model"].isin(MODEL_GROUPS)] if mk == "__pool__"
             else s[s["model"] == mk]).dropna(subset=["somatic_anxiety"]).copy()
        if len(g) < 10:
            out[mk] = (np.nan, np.nan, len(g))
            continue
        z_rt  = (g["mean_log_delib"] - g["mean_log_delib"].mean()) / g["mean_log_delib"].std()
        z_acc = (g["accuracy"]       - g["accuracy"].mean())       / g["accuracy"].std()
        ot = z_acc - z_rt
        rho, p = spearmanr(g["somatic_anxiety"], ot)
        out[mk] = (rho, p, len(g))
    return out

# ── Subject order must match responsibility row order ──
sid_keys = list(winning_params.keys())
sid_str  = [str(s) for s in sid_keys]
max_resp = responsibility.max(axis=1)
argmax_k = responsibility.argmax(axis=1)
second_k = np.argsort(responsibility, axis=1)[:, -2]

# ── Baseline (argmax) ──
baseline_map = {sid_str[i]: GROUP_MAP.get(model_names_choice[argmax_k[i]], "Other")
                for i in range(len(sid_str))}
base = run_ot_from_map(baseline_map)

def _print_block(title, res):
    print("\n" + "="*72)
    print(title)
    print("="*72)
    for mk in GROUPS_ALL:
        r, p, n = res[mk]
        star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
        print(f"  {_label(mk):25s}  rho={r:+.3f}  p={p:.4f} {star:<3s}  n={n}")

_print_block("BASELINE  (argmax responsibility)", base)

# ── A. Decisive-only ──
decisive_mask = max_resp >= TAU
n_drop = int((~decisive_mask).sum())
decisive_map = {sid_str[i]: baseline_map[sid_str[i]]
                for i in range(len(sid_str)) if decisive_mask[i]}
dec = run_ot_from_map(decisive_map)
_print_block(f"A. DECISIVE-ONLY  (max resp >= {TAU}; drop {n_drop}/{len(sid_str)})", dec)

# ── B. Second-best flip ──
flip_map = dict(baseline_map)
for i in range(len(sid_str)):
    if not decisive_mask[i]:
        flip_map[sid_str[i]] = GROUP_MAP.get(model_names_choice[second_k[i]], "Other")
flip = run_ot_from_map(flip_map)
_print_block(f"B. SECOND-BEST FLIP  (borderline subjects swapped to 2nd-best; n_flipped={n_drop})", flip)

# ── C. Monte Carlo ──
rng = np.random.default_rng(SEED)
K = responsibility.shape[1]
mc_rho = {mk: np.full(N_BOOT, np.nan) for mk in GROUPS_ALL}
mc_p   = {mk: np.full(N_BOOT, np.nan) for mk in GROUPS_ALL}
mc_n   = {mk: np.zeros(N_BOOT, dtype=int) for mk in GROUPS_ALL}

for b in range(N_BOOT):
    draws = np.array([rng.choice(K, p=responsibility[i]) for i in range(len(sid_str))])
    mc_map = {sid_str[i]: GROUP_MAP.get(model_names_choice[draws[i]], "Other")
              for i in range(len(sid_str))}
    for mk, (r, p, n) in run_ot_from_map(mc_map).items():
        mc_rho[mk][b] = r
        mc_p[mk][b]   = p
        mc_n[mk][b]   = n

print("\n" + "="*72)
print(f"C. MONTE CARLO  ({N_BOOT} draws from responsibility distribution)")
print("="*72)
print(f"  {'group':25s}  {'rho median':>11s}  {'95% CI':>22s}  {'Pr(p<.05)':>10s}  {'median n':>9s}")
for mk in GROUPS_ALL:
    r = mc_rho[mk][~np.isnan(mc_rho[mk])]
    p = mc_p[mk][~np.isnan(mc_p[mk])]
    n_med = int(np.median(mc_n[mk])) if len(mc_n[mk]) else 0
    if len(r) < 10:
        print(f"  {_label(mk):25s}  (insufficient data, median n={n_med})")
        continue
    ci_lo, ci_hi = np.percentile(r, [2.5, 97.5])
    print(f"  {_label(mk):25s}  {np.median(r):+11.3f}  [{ci_lo:+.3f}, {ci_hi:+.3f}]  {(p < .05).mean():>10.3f}  {n_med:>9d}")

# ── Compact per-scenario summary table ──
print("\n" + "="*72)
print("SUMMARY — rho (p) across scenarios")
print("="*72)
hdr = f"  {'group':25s}  {'baseline':>14s}  {'decisive':>14s}  {'2nd-best':>14s}  {'MC median [CI]':>28s}"
print(hdr)
for mk in GROUPS_ALL:
    def fmt(x):
        r, p, _ = x
        if np.isnan(r): return f"{'--':>14s}"
        star = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""
        return f"{r:+.3f} ({p:.3f}){star}"
    r_mc = mc_rho[mk][~np.isnan(mc_rho[mk])]
    mc_str = "--" if len(r_mc) < 10 else f"{np.median(r_mc):+.3f} [{np.percentile(r_mc,2.5):+.2f},{np.percentile(r_mc,97.5):+.2f}]"
    print(f"  {_label(mk):25s}  {fmt(base[mk]):>14s}  {fmt(dec[mk]):>14s}  {fmt(flip[mk]):>14s}  {mc_str:>28s}")

# ── Visualise MC distribution of rho ──
BG_COLOR = "#FAFBFC"
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5), facecolor=BG_COLOR)

# Left: boxplot of MC rho per group, with baseline overlaid
ax = axL; ax.set_facecolor(BG_COLOR)
vals, positions, colors = [], [], []
MIN_MC = 10
for i, mk in enumerate(GROUPS_ALL):
    r = mc_rho[mk][~np.isnan(mc_rho[mk])]
    if len(r) < MIN_MC: continue
    vals.append(r); positions.append(i)
    colors.append(MODEL_COLORS.get(mk, "#34495E") if mk != "__pool__" else "#34495E")

if len(vals) == 0:
    # No group has enough non-NaN MC samples to plot. Common at very small N — skip silently.
    ax.text(0.5, 0.5,
            f"insufficient MC samples\n(need >= {MIN_MC} non-NaN per group)",
            transform=ax.transAxes, ha="center", va="center",
            color="#888", fontsize=11)
else:
    bp = ax.boxplot(vals, positions=positions, widths=0.55,
                    patch_artist=True, showfliers=False)
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col); patch.set_alpha(0.55)
for i, mk in enumerate(GROUPS_ALL):
    r = base[mk][0]
    if not np.isnan(r):
        ax.plot(i, r, "kD", markersize=9, zorder=5,
                label="argmax baseline" if i == 0 else None)

ax.axhline(0, color="gray", ls="--", alpha=0.5)
ax.set_xticks(range(len(GROUPS_ALL)))
ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_GROUPS] + ["Pool"], fontsize=11)
ax.set_ylabel("Spearman rho  (OT ~ anxiety)", fontsize=11)
ax.set_title(f"MC distribution of rho\n({N_BOOT} draws from HBI responsibility)",
             fontweight="bold", fontsize=11)
ax.legend(loc="best", fontsize=9)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Right: Pr(p<.05) across scenarios
ax = axR; ax.set_facecolor(BG_COLOR)
xs = np.arange(len(GROUPS_ALL))
w = 0.22
scen_labels = ["baseline", "decisive", "2nd-best", "MC mean"]
for j, (lbl, src) in enumerate(zip(scen_labels, [base, dec, flip, None])):
    ys = []
    for mk in GROUPS_ALL:
        if lbl == "MC mean":
            p = mc_p[mk][~np.isnan(mc_p[mk])]
            ys.append((p < .05).mean() if len(p) >= 10 else np.nan)
        else:
            _, pv, _ = src[mk]
            ys.append(1.0 if pv < .05 else 0.0)
    ax.bar(xs + (j - 1.5) * w, ys, w, label=lbl, alpha=0.85, edgecolor="white")

ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax.set_xticks(xs)
ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_GROUPS] + ["Pool"], fontsize=11)
ax.set_ylabel("Pr(p < .05)  /  indicator", fontsize=11)
ax.set_title("Significance robustness across scenarios", fontweight="bold", fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, loc="upper right")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.suptitle("Sensitivity of OT ~ Anxiety finding to HBI assignment uncertainty",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"fig_sensitivity_ot_anxiety_{OUT_PREFIX}.png", dpi=200, bbox_inches="tight", facecolor=BG_COLOR)
plt.show()


---
## 9 · Model and Parameter Recovery

Simulate data from each of the top 5 models, re-fit all 5 candidate models to
the simulated data using the exact same CBM + HBI pipeline, and check whether
each generating model is recovered as the HBI winner.

**Recovery model set (5×5 confusion):**
- `M2_MF_alpha`  — pure MF, learned scaling of cached values
- `M3_MB`        — pure MB, single β
- `M5_MB_stim`   — MB with β per stimulus (Feather vs Triangle)
- `M7_MB_reval`  — MB with β per revaluation phase (Q1 vs Q2)
- `M9_w_stim`    — MB/MF mixture with per-stimulus weight w

**Simulation protocol:** for each generating model we draw 200 simulated
subjects. Each simulated subject inherits a real subject's trial structure
(ev, offer, stimulus, query) and receives parameters sampled *with replacement*
from the empirical distribution of that model's per-subject CBM MAPs — so the
simulated parameter range matches what we actually see in the data.

**Match the pipeline exactly:** CBM individual_fit + HBI + hbi_null (no MLE
shortcuts). This is slow (~30–60 min per filter setting) but is the
methodologically correct benchmark for our real results.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9A  RECOVERY CONFIG: model set, sample sizes, param pools, rng
# ══════════════════════════════════════════════════════════════════════
# Depends on: MODELS, cbm_fits, data_list, N_SUBJ, CBM_DIR, FILTER_SUFFIX, SEED
# All of the above are defined by cells 21–22 (8A–8B).
# ══════════════════════════════════════════════════════════════════════

RECOVERY_MODELS = ["M2_MF_alpha", "M3_MB", "M5_MB_stim", "M7_MB_reval", "M9_w_stim"]
N_SIM_PER_MODEL = 200
RECOVERY_PRIOR_VARIANCE = 6.25  # matches real-data fits

RECOVERY_DIR = f"recovery_output_{OUT_PREFIX}"
os.makedirs(RECOVERY_DIR, exist_ok=True)

# Separate RNG so running recovery doesn't perturb any other stochastic step
recovery_rng = np.random.default_rng(SEED)

# ── Build parameter pools from empirical CBM MAPs ──
# pool[mname] has shape (N_SUBJ, k_m); we draw rows with replacement to build
# realistic synthetic subjects.
param_pools = {}
for mname in RECOVERY_MODELS:
    assert mname in cbm_fits, f"{mname} not in cbm_fits — run cell 22 (CBM fits) first"
    param_pools[mname] = cbm_fits[mname].output.parameters  # (N, k)

print(f"[RECOVERY] Filter: {FILTER_LABEL}")
print(f"[RECOVERY] Dir:    {RECOVERY_DIR}")
print(f"[RECOVERY] Models: {RECOVERY_MODELS}")
print(f"[RECOVERY] N per generating model: {N_SIM_PER_MODEL}")
print(f"[RECOVERY] Param pools:")
for m, p in param_pools.items():
    print(f"  {m:15s}  shape={p.shape}  param_means={np.round(p.mean(axis=0), 3)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9B  SIMULATORS: generate binary choice arrays from each model
# ══════════════════════════════════════════════════════════════════════
# Each simulator takes (params, d, rng) and returns an array of 0/1 choices
# the same length as d["ev"]. The probability expression MUST match the
# log-likelihood in cell 21 (8A) exactly — copy-paste carefully.

def sim_M2_MF_alpha(params, d, rng):
    beta, alpha_raw = params
    alpha = expit(alpha_raw)
    p = expit(beta * (alpha * d["mf_value"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M3_MB(params, d, rng):
    beta = params[0]
    p = expit(beta * (d["ev"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M5_MB_stim(params, d, rng):
    beta_f, beta_t = params
    beta = np.where(d["is_feather"], beta_f, beta_t)
    p = expit(beta * (d["ev"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M7_MB_reval(params, d, rng):
    beta_q1, beta_q2 = params
    beta = np.where(d["is_q1"], beta_q1, beta_q2)
    p = expit(beta * (d["ev"] - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

def sim_M9_w_stim(params, d, rng):
    beta, wf_raw, wt_raw = params
    w = np.where(d["is_feather"], expit(wf_raw), expit(wt_raw))
    v = w * d["ev"] + (1 - w) * d["mf_value"]
    p = expit(beta * (v - d["offer"]))
    return rng.binomial(1, np.clip(p, 1e-8, 1 - 1e-8)).astype(float)

SIMULATORS = {
    "M2_MF_alpha":  sim_M2_MF_alpha,
    "M3_MB":        sim_M3_MB,
    "M5_MB_stim":   sim_M5_MB_stim,
    "M7_MB_reval":  sim_M7_MB_reval,
    "M9_w_stim":    sim_M9_w_stim,
}
print("[RECOVERY] Simulators ready:", list(SIMULATORS.keys()))


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9C  RUN RECOVERY  (~30–60 min for this cell; HBI does most of the waiting)
# ══════════════════════════════════════════════════════════════════════
# For each generating model we:
#   (1) Sample N_SIM_PER_MODEL parameter sets from its empirical pool.
#   (2) For each simulated subject, donor a real trial structure from data_list,
#       then generate binary choices via the matching simulator.
#   (3) Fit ALL 5 candidate models to the simulated dataset via CBM individual_fit.
#   (4) Run HBI (+ hbi_null for PXP) over those 5 candidate fits.
#   (5) Record the HBI-winner for each simulated subject + save the true params.
# Artifacts go to RECOVERY_DIR/gen_<model>/.

import time

recovery_results = {}  # keyed by generating model name

for gen_name in RECOVERY_MODELS:
    t0 = time.time()
    print("\n" + "=" * 72)
    print(f"[RECOVERY] Generating model: {gen_name}")
    print("=" * 72)

    gen_dir = os.path.join(RECOVERY_DIR, f"gen_{gen_name}")
    os.makedirs(gen_dir, exist_ok=True)

    pool = param_pools[gen_name]            # (N_SUBJ, k_gen)
    sim_func = SIMULATORS[gen_name]

    # ── (1) + (2): build sim_data_list + record true params ──
    sim_data_list = []
    true_params = np.zeros((N_SIM_PER_MODEL, pool.shape[1]))
    donor_idx = recovery_rng.integers(0, N_SUBJ, size=N_SIM_PER_MODEL)
    param_idx = recovery_rng.integers(0, pool.shape[0], size=N_SIM_PER_MODEL)

    for i in range(N_SIM_PER_MODEL):
        params = pool[param_idx[i]].copy()
        donor  = data_list[donor_idx[i]]
        d_sim = {k: np.asarray(v).copy() for k, v in donor.items()}
        d_sim["choice"] = sim_func(params, d_sim, recovery_rng)
        sim_data_list.append(d_sim)
        true_params[i] = params

    # ── (3): fit each candidate model to sim_data_list ──
    sim_fits = {}
    for fit_name in RECOVERY_MODELS:
        mspec = MODELS[fit_name]
        k = mspec["k"]
        fname = os.path.join(gen_dir, f"fit_{fit_name}.pkl")
        print(f"  Fitting {fit_name} (k={k})...")
        sim_fits[fit_name] = individual_fit(
            sim_data_list, mspec["func"], np.zeros(k), RECOVERY_PRIOR_VARIANCE,
            fname=fname, config={"verbose": False}
        )

    # ── (4): HBI across the 5 candidates ──
    fit_funcs = [MODELS[m]["func"] for m in RECOVERY_MODELS]
    fit_paths = [os.path.join(gen_dir, f"fit_{m}.pkl") for m in RECOVERY_MODELS]
    hbi_path = os.path.join(gen_dir, "hbi.pkl")
    print(f"  Running HBI over {len(RECOVERY_MODELS)} models...")
    hbi_res = hbi_main(sim_data_list, fit_funcs, fit_paths,
                       fname=hbi_path, config={"verbose": 0, "save_prog": False})
    hbi_res = hbi_null(sim_data_list, hbi_path)

    # ── (5): extract per-subject winners and frequencies ──
    resp = hbi_res.output.responsibility  # (N_SIM_PER_MODEL, 5)
    winners_idx = np.argmax(resp, axis=1)
    winners = [RECOVERY_MODELS[i] for i in winners_idx]

    recovery_results[gen_name] = {
        "true_params":     true_params,
        "responsibility":  resp,
        "winners":         winners,
        "winners_idx":     winners_idx,
        "model_frequency": hbi_res.output.model_frequency,
        "pxp":             hbi_res.output.protected_exceedance_prob,
        "sim_fits":        sim_fits,
        "hbi":             hbi_res,
        "donor_idx":       donor_idx,
        "param_idx":       param_idx,
    }

    # Quick summary printout
    from collections import Counter
    counts = Counter(winners)
    elapsed = time.time() - t0
    print(f"  -> {N_SIM_PER_MODEL} simulated subjects fit in {elapsed/60:.1f} min")
    for m in RECOVERY_MODELS:
        c = counts.get(m, 0)
        marker = "  <- TRUE" if m == gen_name else ""
        print(f"    {m:15s}: {c:3d} winners ({100*c/N_SIM_PER_MODEL:5.1f}%){marker}")

print("\n[RECOVERY] ✅ All generating models done.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9D  MODEL-RECOVERY CONFUSION MATRIX  (row = generating, col = recovered)
# ══════════════════════════════════════════════════════════════════════

K = len(RECOVERY_MODELS)
confusion = np.zeros((K, K), dtype=int)
for i, gen_name in enumerate(RECOVERY_MODELS):
    winners_idx = recovery_results[gen_name]["winners_idx"]
    for j in range(K):
        confusion[i, j] = int(np.sum(winners_idx == j))

confusion_prop = confusion / confusion.sum(axis=1, keepdims=True)

# ── Save CSV ──
conf_csv = os.path.join(RECOVERY_DIR, "confusion_matrix.csv")
import pandas as _pd
_pd.DataFrame(confusion_prop, index=RECOVERY_MODELS, columns=RECOVERY_MODELS)\
    .to_csv(conf_csv, float_format="%.3f")
print(f"[RECOVERY] saved: {conf_csv}")

# ── Plot heatmap ──
fig, ax = plt.subplots(figsize=(7.5, 6.3))
im = ax.imshow(confusion_prop, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(K)); ax.set_yticks(range(K))
ax.set_xticklabels(RECOVERY_MODELS, rotation=35, ha="right")
ax.set_yticklabels(RECOVERY_MODELS)
ax.set_xlabel("Recovered model (HBI winner)", fontweight="bold")
ax.set_ylabel("Generating model", fontweight="bold")
ax.set_title(f"Model Recovery — {FILTER_LABEL}\n(N={N_SIM_PER_MODEL} per generating model)",
             fontweight="bold")
for i in range(K):
    for j in range(K):
        val = confusion_prop[i, j]
        count = confusion[i, j]
        color = "white" if val > 0.45 else "black"
        ax.text(j, i, f"{val:.2f}\n({count})", ha="center", va="center",
                color=color, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.85, label="P(recovered | generating)")
plt.tight_layout()
fig_path = os.path.join(RECOVERY_DIR, "confusion_matrix.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"[RECOVERY] saved: {fig_path}")

# ── Print key recovery diagnostics ──
print("\nDiagonal recoveries (P(recovered = generating)):")
for i, m in enumerate(RECOVERY_MODELS):
    print(f"  {m:15s}: {confusion_prop[i, i]:.2f}")
print(f"\nMean diagonal: {np.mean(np.diag(confusion_prop)):.2f}")
print(f"Worst off-diagonal confusion: {np.max(confusion_prop - np.eye(K) * confusion_prop):.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9E  PARAMETER RECOVERY — recovered vs true, per model & parameter
# ══════════════════════════════════════════════════════════════════════
# For each generating model we pull the FITTED parameters of that same
# model (from the CBM individual_fit it ran against the simulated data of its
# own generator) and scatter them against the TRUE parameters we sampled in.
# This is the cleanest parameter-identifiability check — it ignores the
# model-comparison question (handled by the confusion matrix above) and
# asks just: "if we know the right model, can we recover its parameters?"

from scipy.stats import pearsonr, spearmanr

fig, axes = plt.subplots(len(RECOVERY_MODELS), 4, figsize=(14, 3.1 * len(RECOVERY_MODELS)),
                         squeeze=False)
for row, gen_name in enumerate(RECOVERY_MODELS):
    rec = recovery_results[gen_name]
    true_p = rec["true_params"]
    fit_result = rec["sim_fits"][gen_name]
    fit_p = fit_result.output.parameters
    names = MODELS[gen_name]["names"]
    k = true_p.shape[1]

    for col in range(4):
        ax = axes[row][col]
        if col >= k:
            ax.axis("off")
            continue
        t = true_p[:, col]
        r = fit_p[:, col]
        rho_p, pval_p = pearsonr(t, r)
        rho_s, _     = spearmanr(t, r)
        ax.scatter(t, r, s=14, alpha=0.55, edgecolor="none")
        lims = [min(t.min(), r.min()), max(t.max(), r.max())]
        ax.plot(lims, lims, "k--", lw=0.9, alpha=0.6, label="y = x")
        ax.set_xlabel(f"True {names[col]}")
        ax.set_ylabel(f"Recovered {names[col]}")
        ax.set_title(f"{gen_name} - {names[col]}\nrho_P={rho_p:.2f}  rho_S={rho_s:.2f}", fontsize=9)
        if col == 0:
            ax.text(-0.28, 0.5, gen_name, transform=ax.transAxes,
                    rotation=90, va="center", ha="center",
                    fontsize=11, fontweight="bold")

fig.suptitle(f"Parameter Recovery - {FILTER_LABEL}",
             fontsize=13, fontweight="bold", y=1.005)
plt.tight_layout()
pr_path = os.path.join(RECOVERY_DIR, "param_recovery.png")
plt.savefig(pr_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"[RECOVERY] saved: {pr_path}")

import pandas as _pd
print("\nParameter recovery summary (Pearson r of true vs recovered):")
rows = []
for gen_name in RECOVERY_MODELS:
    rec = recovery_results[gen_name]
    true_p = rec["true_params"]
    fit_p = rec["sim_fits"][gen_name].output.parameters
    names = MODELS[gen_name]["names"]
    for j, pname in enumerate(names):
        r, p = pearsonr(true_p[:, j], fit_p[:, j])
        rows.append({"model": gen_name, "parameter": pname, "r_pearson": r, "p_value": p})
        print(f"  {gen_name:15s} {pname:12s}: r = {r:+.3f}  (p = {p:.3g})")

pr_summary = _pd.DataFrame(rows)
pr_csv = os.path.join(RECOVERY_DIR, "param_recovery_summary.csv")
pr_summary.to_csv(pr_csv, index=False, float_format="%.4f")
print(f"[RECOVERY] saved: {pr_csv}")
